In [16]:
pip install numpy pandas matplotlib scikit-learn scipy shap joblib openpyxl optuna

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------- ----------- 1.6/2.2 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 10.1 MB/s eta 0:00:00

   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]
   -------------------- ------------------- 3/6 [sqlalchemy]


In [12]:
# =====================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

print("=" * 80)
print("FINAL PREPROCESSING PIPELINE (LST lags included, Year in CSV only)")
print("=" * 80)

ml_dir = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed"
input_csv = os.path.join(ml_dir, "Combined.csv")
output_dir = os.path.join(ml_dir, "Spearman")
os.makedirs(output_dir, exist_ok=True)
output_csv   = os.path.join(output_dir, "lagged_ml_ready.csv")
heatmap_path = os.path.join(output_dir, "spearman_lagged_heatmap.png")
# =====================================================================
# STEP 1-2: READ DATA
# =====================================================================
print("\nStep 1-2: Reading dataset...")
df = pd.read_csv(input_csv)
print(f"Loaded {df.shape[0]:,} rows, {df.shape[1]} columns")
print("Columns:", list(df.columns))

# =====================================================================
# STEP 3-7: SORT, LAGS, CLEANING
# =====================================================================
df = df.sort_values(by=["Row_idx", "Col_idx", "Year", "Month"]).reset_index(drop=True)

# Rename SPEI6 → SPEI_6 for consistency
if 'SPEI6' in df.columns:
    df = df.rename(columns={'SPEI6': 'SPEI_6'})

# Create lags for all retained drivers (including LST)
lag_features = ["LST", "SMDI", "SPEI_3", "SPEI_6"]
for feature in lag_features:
    df[f"{feature}_lag1"] = df.groupby(["Row_idx", "Col_idx"])[feature].shift(1)
    df[f"{feature}_lag2"] = df.groupby(["Row_idx", "Col_idx"])[feature].shift(2)

df = df.dropna().reset_index(drop=True)
df = df.drop_duplicates().reset_index(drop=True)

# Columns to exclude from ML predictors scaling
exclude_cols = [
    'Row_idx', 'Col_idx', 'Year', 'Month', 'VCI'
]
predictor_cols = [col for col in df.columns if col not in exclude_cols]

# IQR clipping
for col in predictor_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = np.clip(df[col], Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

# =====================================================================
# STEP 8: NORMALIZATION
# =====================================================================
print("\nStep 8: Normalization...")
scaler = StandardScaler()
df[predictor_cols] = scaler.fit_transform(df[predictor_cols])

# =====================================================================
# STEP 9: SEASONALITY ENCODING
# =====================================================================
print("\nStep 9: Encoding Month...")
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

# =====================================================================
# FINAL ML-READY FILE (Year included here)
# =====================================================================
final_cols = ['Year', 'Month_sin', 'Month_cos'] + predictor_cols + ['VCI']
df_final = df[final_cols].copy()
df_final.to_csv(output_csv, index=False)

print(f"\n✅ ML Ready file saved: {output_csv}")
print(f"Final shape: {df_final.shape}")
print(f"Columns kept: {final_cols}")

# =====================================================================
# VCI CORRELATION SUMMARY (Year excluded from correlation calculation)
# =====================================================================
print("\n" + "=" * 60)
print("VCI CORRELATION SUMMARY (Spearman) – final cleaned set")
print("=" * 60)

# Drop 'Year' temporarily just for correlation calculations
df_for_corr = df_final.drop(columns=['Year'])
corr_with_vci = df_for_corr.corr(method='spearman')['VCI'].sort_values(ascending=False)

for feature, value in corr_with_vci.items():
    if feature != 'VCI':
        strength = "Strong" if abs(value) > 0.4 else "Moderate" if abs(value) > 0.2 else "Weak"
        print(f"{feature:20} : {value:.3f} → {strength}")

# =====================================================================
# SPEARMAN HEATMAP (Year intentionally excluded)
# =====================================================================
print("\nGenerating Publication Quality Heatmap...")
corr_order = [
    'LST', 'LST_lag1', 'LST_lag2',
    'SMDI', 'SMDI_lag1', 'SMDI_lag2',
    'SPEI_3', 'SPEI_3_lag1', 'SPEI_3_lag2',
    'SPEI_6', 'SPEI_6_lag1', 'SPEI_6_lag2',
    'Month_sin', 'Month_cos',
    'VCI'
]
corr_order = [c for c in corr_order if c in df_final.columns]
corr_matrix = df_final[corr_order].corr(method='spearman')

plt.figure(figsize=(15, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask,
            cmap=sns.diverging_palette(230, 20, as_cmap=True),
            vmin=-1.0, vmax=1.0, center=0, square=True, linewidths=0.5,
            annot=True, fmt=".2f", annot_kws={"size": 8, "weight": "bold"},
            cbar_kws={"shrink": 0.75, "label": "Spearman Rank Correlation (ρ)"})

plt.title("Spearman Rank Correlation Matrix",
          fontsize=16, weight="bold", pad=20)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Heatmap saved: {heatmap_path}")
print("\n🎉 All processing completed successfully!")

FINAL PREPROCESSING PIPELINE (LST lags included, Year in CSV only)

Step 1-2: Reading dataset...
Loaded 7,419,160 rows, 9 columns
Columns: ['Row_idx', 'Col_idx', 'Year', 'Month', 'LST', 'SMDI', 'SPEI_3', 'SPEI6', 'VCI']

Step 8: Normalization...

Step 9: Encoding Month...

✅ ML Ready file saved: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\lagged_ml_ready.csv
Final shape: (7270392, 16)
Columns kept: ['Year', 'Month_sin', 'Month_cos', 'LST', 'SMDI', 'SPEI_3', 'SPEI_6', 'LST_lag1', 'LST_lag2', 'SMDI_lag1', 'SMDI_lag2', 'SPEI_3_lag1', 'SPEI_3_lag2', 'SPEI_6_lag1', 'SPEI_6_lag2', 'VCI']

VCI CORRELATION SUMMARY (Spearman) – final cleaned set
SMDI                 : 0.334 → Moderate
SMDI_lag1            : 0.308 → Moderate
SPEI_6               : 0.300 → Moderate
SPEI_6_lag1          : 0.271 → Moderate
SPEI_3               : 0.263 → Moderate
SMDI_lag2            : 0.260 → Moderate
SPEI_3_lag1          : 0.256 → Moderate
SPEI_6_lag2          : 0.212 → Moderate
SPEI_3_lag2     

In [3]:
# =====================================================================
# PUBLICATION-QUALITY FULL SPEARMAN HEATMAP
# VCI is the main response variable
# =====================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------------
# Style settings for maximum visibility & journal quality
# ------------------------------------------------------------------
rcParams['font.family'] = 'DejaVu Sans'
rcParams['axes.labelsize'] = 13
rcParams['xtick.labelsize'] = 11
rcParams['ytick.labelsize'] = 11
rcParams['axes.titlesize'] = 15
rcParams['figure.dpi'] = 300
rcParams['savefig.dpi'] = 600

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
ml_dir = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed"
input_csv = os.path.join(ml_dir, "Combined.csv")
output_dir = os.path.join(ml_dir, "Spearman")
os.makedirs(output_dir, exist_ok=True)
heatmap_path = os.path.join(output_dir, "spearman_full_heatmap_VCI_main.png")

# ------------------------------------------------------------------
# Load & prepare data
# ------------------------------------------------------------------
df = pd.read_csv(input_csv)
df = df.sort_values(by=["Row_idx", "Col_idx", "Year", "Month"]).reset_index(drop=True)

if 'SPEI6' in df.columns:
    df = df.rename(columns={'SPEI6': 'SPEI_6'})

lag_features = ["LST", "SMDI", "SPEI_3", "SPEI_6"]
for feature in lag_features:
    df[f"{feature}_lag1"] = df.groupby(["Row_idx", "Col_idx"])[feature].shift(1)
    df[f"{feature}_lag2"] = df.groupby(["Row_idx", "Col_idx"])[feature].shift(2)

df = df.dropna().reset_index(drop=True)
df = df.drop_duplicates().reset_index(drop=True)

exclude_cols = ['Row_idx', 'Col_idx', 'Year', 'Month', 'VCI']
predictor_cols = [col for col in df.columns if col not in exclude_cols]

for col in predictor_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = np.clip(df[col], Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

scaler = StandardScaler()
df[predictor_cols] = scaler.fit_transform(df[predictor_cols])

df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)

final_cols = ['Month_sin', 'Month_cos'] + predictor_cols + ['VCI']
df_final = df[final_cols].copy()

# ------------------------------------------------------------------
# Correlation matrix – VCI placed FIRST (main response)
# ------------------------------------------------------------------
corr_order = [
    'VCI',                          # ← main response first
    'LST', 'LST_lag1', 'LST_lag2',
    'SMDI', 'SMDI_lag1', 'SMDI_lag2',
    'SPEI_3', 'SPEI_3_lag1', 'SPEI_3_lag2',
    'SPEI_6', 'SPEI_6_lag1', 'SPEI_6_lag2',
    'Month_sin', 'Month_cos'
]
corr_order = [c for c in corr_order if c in df_final.columns]
corr_matrix = df_final[corr_order].corr(method='spearman')

# ------------------------------------------------------------------
# VCI Correlation Summary + Percentage Summary
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("VCI CORRELATION SUMMARY (Spearman) – VCI is the response variable")
print("=" * 65)

corr_with_vci = corr_matrix['VCI'].drop('VCI').sort_values(key=abs, ascending=False)

strong, moderate, weak = 0, 0, 0
total = len(corr_with_vci)

print(f"\n{'Feature':<18} {'ρ':>8}   Strength")
print("-" * 40)
for feature, value in corr_with_vci.items():
    abs_val = abs(value)
    if abs_val > 0.4:
        strength = "Strong"
        strong += 1
    elif abs_val > 0.2:
        strength = "Moderate"
        moderate += 1
    else:
        strength = "Weak"
        weak += 1
    print(f"{feature:<18} {value:>8.3f}   {strength}")

print("\n" + "=" * 65)
print("PERCENTAGE SUMMARY (Correlation Strength with VCI)")
print("=" * 65)
print(f"Strong   (|ρ| > 0.4) : {strong:2d} features  →  {strong/total*100:5.1f}%")
print(f"Moderate (|ρ| > 0.2) : {moderate:2d} features  →  {moderate/total*100:5.1f}%")
print(f"Weak     (|ρ| ≤ 0.2) : {weak:2d} features  →  {weak/total*100:5.1f}%")
print(f"Total predictors     : {total:2d}")
print("=" * 65)

# ------------------------------------------------------------------
# Full (unmasked) heatmap – VCI first
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12.5, 11))

cmap = sns.diverging_palette(220, 20, as_cmap=True)

sns.heatmap(
    corr_matrix,
    cmap=cmap,
    vmin=-1.0, vmax=1.0, center=0,
    square=True,
    linewidths=0.7,
    linecolor='white',
    annot=True,
    fmt=".2f",
    annot_kws={"size": 10, "weight": "medium", "color": "black"},
    cbar_kws={
        "shrink": 0.80,
        "aspect": 28,
        "label": "Spearman Rank Correlation (ρ)",
        "ticks": [-1.0, -0.5, 0.0, 0.5, 1.0]
    },
    ax=ax
)

ax.set_title("Spearman Rank Correlation Matrix\n(VCI as Response Variable)", 
             fontsize=15, fontweight='bold', pad=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig(heatmap_path, dpi=300, bbox_inches="tight", facecolor='white', edgecolor='none')
plt.close()

print(f"\n✅ Full Spearman heatmap saved:\n{heatmap_path}")


VCI CORRELATION SUMMARY (Spearman) – VCI is the response variable

Feature                   ρ   Strength
----------------------------------------
SMDI                  0.334   Moderate
SMDI_lag1             0.308   Moderate
SPEI_6                0.300   Moderate
SPEI_6_lag1           0.271   Moderate
SPEI_3                0.263   Moderate
SMDI_lag2             0.260   Moderate
SPEI_3_lag1           0.256   Moderate
SPEI_6_lag2           0.212   Moderate
SPEI_3_lag2           0.183   Weak
LST_lag1             -0.164   Weak
LST                  -0.159   Weak
LST_lag2             -0.121   Weak
Month_sin             0.019   Weak
Month_cos            -0.001   Weak

PERCENTAGE SUMMARY (Correlation Strength with VCI)
Strong   (|ρ| > 0.4) :  0 features  →    0.0%
Moderate (|ρ| > 0.2) :  8 features  →   57.1%
Weak     (|ρ| ≤ 0.2) :  6 features  →   42.9%
Total predictors     : 14

✅ Full Spearman heatmap saved:
E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\spearman_full_heatm

In [20]:
##XGboost code 
import os
import random
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# Configuration
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

input_csv = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\lagged_ml_ready.csv"

# Load Data
print("Reading dataset...")
df = pd.read_csv(input_csv)
X = df.drop(columns=["VCI"])
y = df["VCI"].copy()

# Normalize target to [0, 1]
target_scaler = MinMaxScaler()
y = target_scaler.fit_transform(y.values.reshape(-1, 1)).ravel()

# 70/30 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, shuffle=True
)

print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

# XGBoost Model (Fixed assignment to 'model')
print("\nTraining XGBoost...")
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.04,
    max_depth=15,
    min_child_weight=7,
    subsample=0.75,
    colsample_bytree=0.75,
    gamma=0.01,
    reg_alpha=0.8,
    reg_lambda=2.0,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
    eval_metric="rmse"
)
model.fit(X_train, y_train)

# Evaluation
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print(f"[XGBoost] Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f}")
print(f"[XGBoost] Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
print(f"[XGBoost] Train MAE: {train_mae:.4f} | Test MAE: {test_mae:.4f}")

Reading dataset...
Training samples : 5,089,274
Testing samples  : 2,181,118

Training XGBoost...
[XGBoost] Train R²: 0.8743 | Test R²: 0.8403
[XGBoost] Train RMSE: 0.0944 | Test RMSE: 0.1063
[XGBoost] Train MAE: 0.0701 | Test MAE: 0.0783


In [26]:
## LightGBM Model 
import os
import random
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

# Configuration
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

input_csv = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\lagged_ml_ready.csv"

# Load Data
print("Reading dataset...")
df = pd.read_csv(input_csv)
X = df.drop(columns=["VCI"])
y = df["VCI"].copy()

# Normalize target to [0, 1]
target_scaler = MinMaxScaler()
y = target_scaler.fit_transform(y.values.reshape(-1, 1)).ravel()

# 70/30 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, shuffle=True
)

print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

# ====================== LightGBM Model ======================
print("\nTraining LightGBM...")
model = LGBMRegressor(
    n_estimators=1400,
    learning_rate=0.04,
    max_depth=15,            
    num_leaves=2048,         # Increased from 200 to give the model proper capacity
    min_child_samples=50,    # Safer value for large datasets to prevent noisy splits
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.01,
    reg_lambda=0.2,          
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)
model.fit(X_train, y_train)

# Evaluation
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print(f"\n[LightGBM] Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f}")
print(f"[LightGBM] Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
print(f"[LightGBM] Train MAE: {train_mae:.4f} | Test MAE: {test_mae:.4f}")

Reading dataset...
Training samples : 5,089,274
Testing samples  : 2,181,118

Training LightGBM...

[LightGBM] Train R²: 0.8528 | Test R²: 0.8271
[LightGBM] Train RMSE: 0.1022 | Test RMSE: 0.1107
[LightGBM] Train MAE: 0.0758 | Test MAE: 0.0818


In [ ]:
Training XGBoost...
[XGBoost] Train R²: 0.8743 | Test R²: 0.8403
[XGBoost] Train RMSE: 0.0944 | Test RMSE: 0.1063
[XGBoost] Train MAE: 0.0701 | Test MAE: 0.0783

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

# Configuration
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

input_csv = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\lagged_ml_ready.csv"

# Load Data
print("Reading dataset...")
df = pd.read_csv(input_csv)
X = df.drop(columns=["VCI"])
y = df["VCI"].copy()

# Normalize target to [0, 1]
target_scaler = MinMaxScaler()
y = target_scaler.fit_transform(y.values.reshape(-1, 1)).ravel()

# 70/30 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, shuffle=True
)

print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

# ====================== Random Forest Model ======================
print("\nTraining Random Forest...")
model = RandomForestRegressor(
    n_estimators=500,        # 100 trees is standard for RF (unlike boosting, RF doesn't need 1000)
    max_depth=17,            # Matches the depth you used in XGBoost/LightGBM
    min_samples_split=10,    # Regularization to prevent leaf fragmentation
    min_samples_leaf=5,      # Regularization for stability
    random_state=RANDOM_STATE,
    n_jobs=-1,               # Uses all available CPU cores
    verbose=1                # Prints progress during training
)
model.fit(X_train, y_train)

# Evaluation
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print(f"\n[Random Forest] Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f}")
print(f"[Random Forest] Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
print(f"[Random Forest] Train MAE: {train_mae:.4f} | Test MAE: {test_mae:.4f}")

Reading dataset...
Training samples : 5,089,274
Testing samples  : 2,181,118

Training Random Forest with expanded feature access...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:  7.8min
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed: 47.6min


In [12]:
import os
import random
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

# Configuration
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

input_csv = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\lagged_ml_ready.csv"

# Load Data
print("Reading dataset...")
df = pd.read_csv(input_csv)
X = df.drop(columns=["VCI"])
y = df["VCI"].copy()

# Normalize target to [0, 1]
target_scaler = MinMaxScaler()
y = target_scaler.fit_transform(y.values.reshape(-1, 1)).ravel()

# 70/30 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, shuffle=True
)

print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

print("\nTraining Optimized CatBoost...")
model = CatBoostRegressor(
    iterations=1500,          # Give it enough room to learn
    learning_rate=0.04,
    depth=10,                 # Safe, high-capacity depth (1024 leaves)
    l2_leaf_reg=1.5,          # Reduced regularization to let it capture patterns
    loss_function="RMSE",
    eval_metric="RMSE",
    early_stopping_rounds=100,
    task_type="CPU",
    thread_count=-1,
    random_seed=RANDOM_STATE,
    verbose=100
)

model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=100
)
model.fit(X_train, y_train)

# Evaluation
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
train_mae = mean_absolute_error(y_train, train_pred)
test_mae = mean_absolute_error(y_test, test_pred)

print(f"\n[CatBoost] Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f}")
print(f"[CatBoost] Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
print(f"[CatBoost] Train MAE: {train_mae:.4f} | Test MAE: {test_mae:.4f}")

Reading dataset...
Training samples : 5,089,274
Testing samples  : 2,181,118

Training Optimized CatBoost...
0:	learn: 0.2625947	test: 0.2625124	best: 0.2625124 (0)	total: 679ms	remaining: 16m 58s
100:	learn: 0.1830226	test: 0.1830257	best: 0.1830257 (100)	total: 1m 6s	remaining: 15m 22s
200:	learn: 0.1734984	test: 0.1735320	best: 0.1735320 (200)	total: 2m 11s	remaining: 14m 8s
300:	learn: 0.1683871	test: 0.1684659	best: 0.1684659 (300)	total: 3m 10s	remaining: 12m 38s
400:	learn: 0.1648392	test: 0.1649687	best: 0.1649687 (400)	total: 4m 10s	remaining: 11m 27s
500:	learn: 0.1619970	test: 0.1621753	best: 0.1621753 (500)	total: 5m 12s	remaining: 10m 22s
600:	learn: 0.1595905	test: 0.1598078	best: 0.1598078 (600)	total: 6m 13s	remaining: 9m 18s
700:	learn: 0.1576642	test: 0.1579244	best: 0.1579244 (700)	total: 7m 17s	remaining: 8m 18s
800:	learn: 0.1559810	test: 0.1562785	best: 0.1562785 (800)	total: 8m 20s	remaining: 7m 17s
900:	learn: 0.1545465	test: 0.1548867	best: 0.1548867 (900)	tota

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

# ====================== Configuration ======================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

input_csv = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\lagged_ml_ready.csv"
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\scatterplot"
os.makedirs(output_dir, exist_ok=True)

# ====================== Load Data ======================
print("Reading dataset...")
df = pd.read_csv(input_csv)
print(f"Dataset size: {len(df):,} rows")

X = df.drop(columns=["VCI"])
y = df["VCI"].copy()

# Normalize target to [0, 1]
target_scaler = MinMaxScaler()
y = target_scaler.fit_transform(y.values.reshape(-1, 1)).ravel()

# 70/30 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, shuffle=True
)

# ====================== Define Models ======================
models = {
    "XGBoost": XGBRegressor(
        n_estimators=1000,
        learning_rate=0.04,
        max_depth=15,
        min_child_weight=7,
        subsample=0.75,
        colsample_bytree=0.75,
        gamma=0.01,
        reg_alpha=0.8,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
        eval_metric="rmse"
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=1400,
        learning_rate=0.04,
        max_depth=15,            
        num_leaves=2048,         
        min_child_samples=50,    
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.01,
        reg_lambda=0.2,          
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=150,        
        max_depth=17,            
        min_samples_split=10,    
        min_samples_leaf=5,      
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

# Train and predict
train_preds = {}
test_preds = {}

print("\nTraining models and predicting...")
for name, model in models.items():
    print(f" -> Training {name}...")
    model.fit(X_train, y_train)
    train_preds[name] = model.predict(X_train)
    test_preds[name] = model.predict(X_test)

# Setup Subfigure Labels mapping (clean letters: a, b, c)
labels_map = {
    "XGBoost": "a",
    "LightGBM": "b",
    "RandomForest": "c"
}

# Sample indices for rendering efficiency (100k points)
train_sample_size = min(100_000, len(y_train))
train_indices = np.random.RandomState(RANDOM_STATE).choice(len(y_train), size=train_sample_size, replace=False)
y_train_sample = y_train[train_indices]

test_sample_size = min(100_000, len(y_test))
test_indices = np.random.RandomState(RANDOM_STATE).choice(len(y_test), size=test_sample_size, replace=False)
y_test_sample = y_test[test_indices]

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["figure.dpi"] = 300
box_props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='lightgray')

# ======================================================================
# 1. INDIVIDUAL PLOTS FOR EACH MODEL (Train & Test side-by-side with labels)
# ======================================================================
print("\nGenerating individual model scatterplots...")
for name in models.keys():
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor='white')
    tag = labels_map[name]
    
    # --- Training Subplot ---
    ax1 = axes[0]
    tr_preds_sample = train_preds[name][train_indices]
    xy_tr = np.vstack([y_train_sample, tr_preds_sample])
    density_tr = gaussian_kde(xy_tr)(xy_tr)
    idx_tr = density_tr.argsort()
    
    ax1.scatter(y_train_sample[idx_tr], tr_preds_sample[idx_tr], c=density_tr[idx_tr], cmap='Blues', s=4, alpha=0.6, linewidths=0)
    ax1.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)
    
    tr_r2 = r2_score(y_train, train_preds[name])
    tr_rmse = np.sqrt(mean_squared_error(y_train, train_preds[name]))
    tr_mae = mean_absolute_error(y_train, train_preds[name])
    
    tr_text = f"Train Metrics:\nR²: {tr_r2:.4f}\nRMSE: {tr_rmse:.4f}\nMAE: {tr_mae:.4f}"
    ax1.text(0.04, 0.96, tr_text, transform=ax1.transAxes, fontsize=10, bbox=box_props, verticalalignment='top')
    ax1.set_xlim([-0.05, 1.05]); ax1.set_ylim([-0.05, 1.05])
    ax1.set_xlabel('Actual VCI', fontsize=11)
    ax1.set_ylabel('Predicted VCI', fontsize=11)
    ax1.set_title(f'({tag}1) {name} - Training Set', fontsize=12, fontweight='bold', pad=10)
    ax1.grid(True, linestyle=':', alpha=0.45)
    ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

    # --- Testing Subplot ---
    ax2 = axes[1]
    te_preds_sample = test_preds[name][test_indices]
    xy_te = np.vstack([y_test_sample, te_preds_sample])
    density_te = gaussian_kde(xy_te)(xy_te)
    idx_te = density_te.argsort()
    
    ax2.scatter(y_test_sample[idx_te], te_preds_sample[idx_te], c=density_te[idx_te], cmap='Blues', s=4, alpha=0.6, linewidths=0)
    ax2.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)
    
    te_r2 = r2_score(y_test, test_preds[name])
    te_rmse = np.sqrt(mean_squared_error(y_test, test_preds[name]))
    te_mae = mean_absolute_error(y_test, test_preds[name])
    
    te_text = f"Test Metrics:\nR²: {te_r2:.4f}\nRMSE: {te_rmse:.4f}\nMAE: {te_mae:.4f}"
    ax2.text(0.04, 0.96, te_text, transform=ax2.transAxes, fontsize=10, bbox=box_props, verticalalignment='top')
    ax2.set_xlim([-0.05, 1.05]); ax2.set_ylim([-0.05, 1.05])
    ax2.set_xlabel('Actual VCI', fontsize=11)
    ax2.set_ylabel('Predicted VCI', fontsize=11)
    ax2.set_title(f'({tag}2) {name} - Test Set', fontsize=12, fontweight='bold', pad=10)
    ax2.grid(True, linestyle=':', alpha=0.45)
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

    plt.tight_layout()
    indiv_path = os.path.join(output_dir, f"{tag}_{name}_Scatterplot.png")
    plt.savefig(indiv_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {indiv_path}")

# ======================================================================
# 2. COMBINED 2x3 GRID PLOT (All Models Together with Labels a, b, c)
# ======================================================================
print("\nGenerating combined 2x3 multi-model grid scatterplot...")
fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor='white')
model_keys = list(models.keys())

# Row 0: Training Plots (a, b, c)
for col_idx, name in enumerate(model_keys):
    ax = axes[0, col_idx]
    tag = labels_map[name]
    preds_sample = train_preds[name][train_indices]
    
    xy = np.vstack([y_train_sample, preds_sample])
    density = gaussian_kde(xy)(xy)
    idx = density.argsort()
    
    ax.scatter(y_train_sample[idx], preds_sample[idx], c=density[idx], cmap='Blues', s=4, alpha=0.6, linewidths=0)
    ax.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)
    
    r2 = r2_score(y_train, train_preds[name])
    rmse = np.sqrt(mean_squared_error(y_train, train_preds[name]))
    mae = mean_absolute_error(y_train, train_preds[name])
    
    text_str = f"Train Metrics:\nR²: {r2:.4f}\nRMSE: {rmse:.4f}\nMAE: {mae:.4f}"
    ax.text(0.04, 0.96, text_str, transform=ax.transAxes, fontsize=10, bbox=box_props, verticalalignment='top')
    ax.set_xlim([-0.05, 1.05]); ax.set_ylim([-0.05, 1.05])
    ax.set_ylabel('Predicted VCI', fontsize=11)
    ax.set_title(f'({tag}) {name} (Training)', fontsize=12, fontweight='bold', pad=10)
    ax.grid(True, linestyle=':', alpha=0.45)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Row 1: Testing Plots (a, b, c)
for col_idx, name in enumerate(model_keys):
    ax = axes[1, col_idx]
    tag = labels_map[name]
    preds_sample = test_preds[name][test_indices]
    
    xy = np.vstack([y_test_sample, preds_sample])
    density = gaussian_kde(xy)(xy)
    idx = density.argsort()
    
    ax.scatter(y_test_sample[idx], preds_sample[idx], c=density[idx], cmap='Blues', s=4, alpha=0.6, linewidths=0)
    ax.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)
    
    r2 = r2_score(y_test, test_preds[name])
    rmse = np.sqrt(mean_squared_error(y_test, test_preds[name]))
    mae = mean_absolute_error(y_test, test_preds[name])
    
    text_str = f"Test Metrics:\nR²: {r2:.4f}\nRMSE: {rmse:.4f}\nMAE: {mae:.4f}"
    ax.text(0.04, 0.96, text_str, transform=ax.transAxes, fontsize=10, bbox=box_props, verticalalignment='top')
    ax.set_xlim([-0.05, 1.05]); ax.set_ylim([-0.05, 1.05])
    ax.set_xlabel('Actual VCI', fontsize=11)
    ax.set_ylabel('Predicted VCI', fontsize=11)
    ax.set_title(f'({tag}) {name} (Testing)', fontsize=12, fontweight='bold', pad=10)
    ax.grid(True, linestyle=':', alpha=0.45)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
combined_path = os.path.join(output_dir, "All_Models_Combined_Grid_a_b_c.png")
plt.savefig(combined_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved combined grid: {combined_path}")

print("=" * 75)
print(f"All 300 DPI scatterplots successfully saved to:\n{output_dir}")
print("=" * 75)

Reading dataset...
Dataset size: 7,270,392 rows

Training models and predicting...
 -> Training XGBoost...
 -> Training LightGBM...
 -> Training RandomForest...
Saved: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\scatterplot\a_XGBoost_Scatterplot.png
Saved: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\scatterplot\b_LightGBM_Scatterplot.png
Saved: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\scatterplot\c_RandomForest_Scatterplot.png

Generating combined 2x3 multi-model grid scatterplot...


In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import pearsonr, gaussian_kde
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

# ======================================================================
# CONFIGURATION & DIRECTORY SETUP
# ======================================================================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

TEST_SIZE = 0.30
input_csv = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\lagged_ml_ready.csv"
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\ALL"
os.makedirs(output_dir, exist_ok=True)

# Consistent professional color palette for the 3 models
model_colors = {
    "XGBoost": "#1f77b4",       # Muted Blue
    "LightGBM": "#2ca02c",      # Muted Green
    "RandomForest": "#d62728"   # Muted Red
}

# ======================================================================
# STEP 1: LOAD & PREPROCESS DATA
# ======================================================================
print("Reading dataset...")
df = pd.read_csv(input_csv)
print(f"Dataset size: {len(df):,} rows")

X = df.drop(columns=["VCI"])
y = df["VCI"].copy()

# Normalize target to [0, 1]
target_scaler = MinMaxScaler()
y = target_scaler.fit_transform(y.values.reshape(-1, 1)).ravel()

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

# ======================================================================
# STEP 2: DEFINE MODELS
# ======================================================================
models = {
    "XGBoost": XGBRegressor(
        n_estimators=1000,
        learning_rate=0.04,
        max_depth=15,
        min_child_weight=7,
        subsample=0.75,
        colsample_bytree=0.75,
        gamma=0.01,
        reg_alpha=0.8,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
        eval_metric="rmse"
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=1400,
        learning_rate=0.04,
        max_depth=15,            
        num_leaves=2048,         
        min_child_samples=50,    
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.01,
        reg_lambda=0.2,          
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=150,        
        max_depth=17,            
        min_samples_split=10,    
        min_samples_leaf=5,      
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

# ======================================================================
# STEP 3: TRAIN MODELS & COLLECT RESULTS
# ======================================================================
model_results = {}
print("\nTraining models and collecting metrics...")

for name, model in models.items():
    print(f" → Training {name}...")
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred  = model.predict(X_test)

    train_r2   = r2_score(y_train, train_pred)
    test_r2    = r2_score(y_test, test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse  = np.sqrt(mean_squared_error(y_test, test_pred))
    train_mae  = mean_absolute_error(y_train, train_pred)
    test_mae   = mean_absolute_error(y_test, test_pred)
    pearson_r, _ = pearsonr(y_test, test_pred)

    model_results[name] = {
        "model": model,
        "train_pred": train_pred,
        "test_pred": test_pred,
        "train_r2": train_r2,
        "test_r2": test_r2,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "pearson_r": pearson_r,
        "train_res": y_train - train_pred,
        "test_res": y_test - test_pred
    }
    print(f"    Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f}")

# Setup global plotting parameters for journal publishing
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["figure.dpi"] = 300
box_props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='lightgray')

# ======================================================================
# STEP 4: TAYLOR DIAGRAM (CORRECTED RMSD)
# ======================================================================
print("\nGenerating Taylor Diagram...")

std_ref = np.std(y_test)
model_stds = [np.std(res["test_pred"]) for res in model_results.values()]
max_std = max(model_stds + [std_ref]) * 1.25

fig = plt.figure(figsize=(8.5, 8), facecolor="white")
ax = fig.add_subplot(111, polar=True)

ax.set_thetamin(0)
ax.set_thetamax(90)
ax.set_ylim(0, max_std)

# Correlation labels
corr_labels = [0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99, 1.0]
theta_labels = np.degrees(np.arccos(corr_labels))
ax.set_thetagrids(theta_labels, labels=[f"{c:.2f}" for c in corr_labels], fontsize=10)
ax.set_ylabel("Standard Deviation", fontsize=12, labelpad=40)

# Correct RMSD contours
theta = np.linspace(0, np.pi/2, 400)
r = np.linspace(0, max_std, 400)
THETA, R = np.meshgrid(theta, r)
RMSD = np.sqrt(std_ref**2 + R**2 - 2 * std_ref * R * np.cos(THETA))

levels = np.linspace(0.05, max_std * 0.9, 7)
cs = ax.contour(THETA, R, RMSD, levels=levels, colors="gray",
                linestyles="--", linewidths=0.9, alpha=0.8)
ax.clabel(cs, inline=True, fontsize=8, fmt="%.2f")

# Observed reference point
ax.plot(0, std_ref, marker="*", markersize=18, color="red",
        label="Observed VCI", zorder=10)

# Model points
for name, res in model_results.items():
    pred_std = np.std(res["test_pred"])
    r_val = res["pearson_r"]
    theta_model = np.arccos(np.clip(r_val, -1, 1))
    ax.plot(theta_model, pred_std,
            marker="o", markersize=11, linestyle="None",
            color=model_colors[name],
            label=f"{name}\nr = {r_val:.3f} | R² = {res['test_r2']:.3f}")

ax.set_title("Taylor Diagram – VCI Prediction Models", fontsize=14, fontweight="bold", pad=28)
ax.legend(loc="upper right", bbox_to_anchor=(1.42, 1.12), fontsize=9, frameon=True)

plt.tight_layout()
taylor_path = os.path.join(output_dir, "Taylor_Diagram_VCI_Models.png")
plt.savefig(taylor_path, dpi=300, bbox_inches="tight")
plt.close()
print(f" Saved: {taylor_path}")

# ======================================================================
# STEP 5: RESIDUAL ERROR DISTRIBUTION (ALL MODELS)
# ======================================================================
print("\nGenerating Residual Error Distribution plots...")

all_res = np.concatenate([res["test_res"] for res in model_results.values()])
x_min, x_max = all_res.min() - 0.05, all_res.max() + 0.05

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, (name, res) in zip(axes, model_results.items()):
    color = model_colors[name]
    residuals = res["test_res"]

    ax.hist(residuals, bins=50, density=True, alpha=0.35, color=color,
            edgecolor="black", linewidth=0.5, label="Residuals")

    kde = stats.gaussian_kde(residuals)
    x_grid = np.linspace(x_min, x_max, 400)
    ax.plot(x_grid, kde(x_grid), color=color, linewidth=2.5, label="KDE")

    ax.axvline(0, color="black", linestyle="--", linewidth=1.2)

    mean_err = np.mean(residuals)
    std_err  = np.std(residuals)
    rmse     = np.sqrt(np.mean(residuals**2))

    ax.text(0.95, 0.95,
            f"Mean = {mean_err:.5f}\nStd = {std_err:.4f}\nRMSE = {rmse:.4f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.85, edgecolor="gray"))

    ax.set_xlim(x_min, x_max)
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xlabel("Residual Error\n(Observed VCI − Predicted VCI)", fontsize=11)
    ax.grid(linestyle=":", alpha=0.5)

axes[0].set_ylabel("Frequency Density", fontsize=11)
fig.suptitle("Residual Error Distribution Comparison of Machine Learning Models", fontsize=15, fontweight="bold", y=1.03)

plt.tight_layout()
resid_path = os.path.join(output_dir, "All_Models_Residual_Distribution_Comparison.png")
plt.savefig(resid_path, dpi=300, bbox_inches="tight")
plt.close()
print(f" Saved: {resid_path}")

# ======================================================================
# STEP 6: INDIVIDUAL BOXPLOTS (TRAIN vs TEST)
# ======================================================================
print("\nGenerating individual Train vs Test absolute error boxplots...")

for name, res in model_results.items():
    fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")

    abs_train = np.abs(res["train_res"])
    abs_test  = np.abs(res["test_res"])

    bp = ax.boxplot([abs_train, abs_test],
                    labels=["Train Error", "Test Error"],
                    patch_artist=True, showfliers=False, widths=0.45,
                    medianprops=dict(color="black", linewidth=1.8))

    for patch in bp["boxes"]:
        patch.set_facecolor(model_colors[name])
        patch.set_alpha(0.75)
        patch.set_edgecolor("black")
        patch.set_linewidth(1.2)

    ax.set_title(f"{name} − Train vs Test Absolute Error", fontsize=11, fontweight="bold", pad=12)
    ax.set_ylabel("Absolute Residual Error (|Actual − Predicted|)", fontsize=10)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    path = os.path.join(output_dir, f"{name.replace(' ', '_')}_Train_vs_Test_Boxplot.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f" Saved: {path}")

# ======================================================================
# STEP 7: COMPARATIVE TEST-SET BOXPLOT ACROSS MODELS
# ======================================================================
print("\nGenerating comparative test-set boxplot...")

fig, ax = plt.subplots(figsize=(8, 5.5), facecolor="white")

all_test_errors = [np.abs(model_results[m]["test_res"]) for m in models.keys()]

bp = ax.boxplot(all_test_errors, labels=list(models.keys()),
                patch_artist=True, showfliers=False, widths=0.45,
                medianprops=dict(color="black", linewidth=1.8))

for patch, name in zip(bp["boxes"], models.keys()):
    patch.set_facecolor(model_colors[name])
    patch.set_alpha(0.80)
    patch.set_edgecolor("black")
    patch.set_linewidth(1.2)

ax.set_title("Test Set Absolute Residual Error Comparison Across Models", fontsize=12, fontweight="bold", pad=12)
ax.set_ylabel("Absolute Residual Error (|Actual − Predicted|)", fontsize=11)
ax.grid(True, linestyle=":", alpha=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
comp_box_path = os.path.join(output_dir, "All_Models_Test_Boxplot_Comparison.png")
plt.savefig(comp_box_path, dpi=300, bbox_inches="tight")
plt.close()
print(f" Saved: {comp_box_path}")

# ======================================================================
# STEP 8: SCATTERPLOTS (DODGERBLUE SOLID COLOR + RED 1:1 LINE)
# ======================================================================
print("\nGenerating Scatterplots (Individual & Combined Master Grid)...")

# Sampling 100k points for efficient scatter plot rendering
train_sample_size = min(100_000, len(y_train))
train_indices = np.random.RandomState(RANDOM_STATE).choice(len(y_train), size=train_sample_size, replace=False)
y_train_sample = y_train[train_indices]

test_sample_size = min(100_000, len(y_test))
test_indices = np.random.RandomState(RANDOM_STATE).choice(len(y_test), size=test_sample_size, replace=False)
y_test_sample = y_test[test_indices]

model_names = list(models.keys())

# 8A. Individual Model Scatterplots (Train vs Test side-by-side)
for name in model_names:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor='white')
    
    # Training Subplot
    ax1 = axes[0]
    tr_preds_sample = model_results[name]["train_pred"][train_indices]
    ax1.scatter(y_train_sample, tr_preds_sample, color='dodgerblue', alpha=0.25, s=4, linewidths=0)
    ax1.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)
    
    tr_text = f"Train Metrics:\nR²: {model_results[name]['train_r2']:.4f}\nRMSE: {model_results[name]['train_rmse']:.4f}\nMAE: {model_results[name]['train_mae']:.4f}"
    ax1.text(0.04, 0.96, tr_text, transform=ax1.transAxes, fontsize=10, bbox=box_props, verticalalignment='top')
    
    ax1.set_xlim([-0.05, 1.05]); ax1.set_ylim([-0.05, 1.05])
    ax1.set_xlabel('Actual VCI', fontsize=11)
    ax1.set_ylabel('Predicted VCI', fontsize=11)
    ax1.set_title(f'{name} - Training Set', fontsize=12, fontweight='bold', pad=10)
    ax1.grid(True, linestyle=':', alpha=0.45)
    ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

    # Testing Subplot
    ax2 = axes[1]
    te_preds_sample = model_results[name]["test_pred"][test_indices]
    ax2.scatter(y_test_sample, te_preds_sample, color='dodgerblue', alpha=0.25, s=4, linewidths=0)
    ax2.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)
    
    te_text = f"Test Metrics:\nR²: {model_results[name]['test_r2']:.4f}\nRMSE: {model_results[name]['test_rmse']:.4f}\nMAE: {model_results[name]['test_mae']:.4f}"
    ax2.text(0.04, 0.96, te_text, transform=ax2.transAxes, fontsize=10, bbox=box_props, verticalalignment='top')
    
    ax2.set_xlim([-0.05, 1.05]); ax2.set_ylim([-0.05, 1.05])
    ax2.set_xlabel('Actual VCI', fontsize=11)
    ax2.set_ylabel('Predicted VCI', fontsize=11)
    ax2.set_title(f'{name} - Testing Set', fontsize=12, fontweight='bold', pad=10)
    ax2.grid(True, linestyle=':', alpha=0.45)
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

    plt.tight_layout()
    indiv_path = os.path.join(output_dir, f"{name}_Train_Test_Scatterplot.png")
    plt.savefig(indiv_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f" Saved: {indiv_path}")

# 8B. Combined Master 2x3 Grid Scatterplot
fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor='white')

# Row 0: Training Plots
for col_idx, name in enumerate(model_names):
    ax = axes[0, col_idx]
    tr_preds_sample = model_results[name]["train_pred"][train_indices]
    ax.scatter(y_train_sample, tr_preds_sample, color='dodgerblue', alpha=0.25, s=4, linewidths=0)
    ax.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)
    
    text_str = f"Train Metrics:\nR²: {model_results[name]['train_r2']:.4f}\nRMSE: {model_results[name]['train_rmse']:.4f}\nMAE: {model_results[name]['train_mae']:.4f}"
    ax.text(0.04, 0.96, text_str, transform=ax.transAxes, fontsize=10, bbox=box_props, verticalalignment='top')
    
    ax.set_xlim([-0.05, 1.05]); ax.set_ylim([-0.05, 1.05])
    ax.set_ylabel('Predicted VCI', fontsize=11)
    ax.set_title(f'{name} (Training)', fontsize=12, fontweight='bold', pad=10)
    ax.grid(True, linestyle=':', alpha=0.45)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Row 1: Testing Plots
for col_idx, name in enumerate(model_names):
    ax = axes[1, col_idx]
    te_preds_sample = model_results[name]["test_pred"][test_indices]
    ax.scatter(y_test_sample, te_preds_sample, color='dodgerblue', alpha=0.25, s=4, linewidths=0)
    ax.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)
    
    text_str = f"Test Metrics:\nR²: {model_results[name]['test_r2']:.4f}\nRMSE: {model_results[name]['test_rmse']:.4f}\nMAE: {model_results[name]['test_mae']:.4f}"
    ax.text(0.04, 0.96, text_str, transform=ax.transAxes, fontsize=10, bbox=box_props, verticalalignment='top')
    
    ax.set_xlim([-0.05, 1.05]); ax.set_ylim([-0.05, 1.05])
    ax.set_xlabel('Actual VCI', fontsize=11)
    ax.set_ylabel('Predicted VCI', fontsize=11)
    ax.set_title(f'{name} (Testing)', fontsize=12, fontweight='bold', pad=10)
    ax.grid(True, linestyle=':', alpha=0.45)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
grid_path = os.path.join(output_dir, "All_Models_Combined_Train_Test_Grid.png")
plt.savefig(grid_path, dpi=300, bbox_inches='tight')
plt.close()
print(f" Saved: {grid_path}")

print("\n" + "=" * 75)
print(f"ALL EVALUATION METRICS & JOURNAL-GRADE FIGURES SAVED SUCCESSFULLY TO:\n{output_dir}")
print("=" * 75)

Reading dataset...
Dataset size: 7,270,392 rows
Training samples : 5,089,274
Testing samples  : 2,181,118

Training models and collecting metrics...
 → Training XGBoost...
    Train R²: 0.8743 | Test R²: 0.8403
 → Training LightGBM...
    Train R²: 0.7598 | Test R²: 0.7427

Generating Taylor Diagram...
 Saved: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\ALL\Taylor_Diagram_VCI_Models.png

Generating Residual Error Distribution plots...


In [ ]:
## 5-Fold CV + proper XGBoost Learning Curve (same training)
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
warnings.filterwarnings("ignore")

# ====================== Configuration ======================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

input_csv = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\lagged_ml_ready.csv"
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\Results"
os.makedirs(output_dir, exist_ok=True)

# ====================== Load Data ======================
print("Reading dataset...")
df = pd.read_csv(input_csv)
print(f"Dataset size: {len(df):,} rows")

X = df.drop(columns=["VCI"])
y = df["VCI"].copy()

target_scaler = MinMaxScaler()
y = target_scaler.fit_transform(y.values.reshape(-1, 1)).ravel()

X = X.to_numpy()
y = y

# ====================== Model Factory ======================
models_cv = {
    "XGBoost": lambda: XGBRegressor(
        n_estimators=1000,
        learning_rate=0.04,
        max_depth=15,
        min_child_weight=7,
        subsample=0.75,
        colsample_bytree=0.75,
        gamma=0.01,
        reg_alpha=0.8,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
        eval_metric="mae"          # needed for learning curve
    ),
    "LightGBM": lambda: LGBMRegressor(
        n_estimators=1400,
        learning_rate=0.04,
        max_depth=15,
        num_leaves=2048,
        min_child_samples=50,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.01,
        reg_lambda=0.2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    ),
    "RandomForest": lambda: RandomForestRegressor(
        n_estimators=150,
        max_depth=17,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

model_colors = {
    "XGBoost": "#1f77b4",
    "LightGBM": "#2ca02c",
    "RandomForest": "#d62728"
}

# ====================== 5-Fold Cross-Validation ======================
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {m: {"val_mae": [], "val_rmse": [], "val_r2": []} for m in models_cv}

# Storage for XGBoost learning curves (one list per fold)
xgb_train_maes = []
xgb_val_maes   = []

print("\nRunning 5-Fold Cross-Validation...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    for name, model_fn in models_cv.items():
        print(f" -> Training {name} on Fold {fold}/5...")
        model = model_fn()

        if name == "XGBoost":
            # Fit WITH eval_set so we get the learning curve
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_tr, y_tr), (X_val, y_val)],
                verbose=False
            )
            # Collect MAE history for this fold
            eval_res = model.evals_result()
            xgb_train_maes.append(eval_res["validation_0"]["mae"])
            xgb_val_maes.append(eval_res["validation_1"]["mae"])
        else:
            # Normal fit for LightGBM & RandomForest
            model.fit(X_tr, y_tr)

        # Common metrics (all models)
        val_pred = model.predict(X_val)
        cv_results[name]["val_mae"].append(mean_absolute_error(y_val, val_pred))
        cv_results[name]["val_rmse"].append(np.sqrt(mean_squared_error(y_val, val_pred)))
        cv_results[name]["val_r2"].append(r2_score(y_val, val_pred))

    print(f" Fold {fold}/5 finished.\n")

# ====================== Summary Table ======================
cv_summary = []
for name in models_cv:
    mae  = np.array(cv_results[name]["val_mae"])
    rmse = np.array(cv_results[name]["val_rmse"])
    r2   = np.array(cv_results[name]["val_r2"])
    cv_summary.append({
        "Model": name,
        "CV MAE Mean": np.mean(mae),
        "CV MAE Std": np.std(mae),
        "CV RMSE Mean": np.mean(rmse),
        "CV RMSE Std": np.std(rmse),
        "CV R2 Mean": np.mean(r2),
        "CV R2 Std": np.std(r2)
    })

cv_df = pd.DataFrame(cv_summary)
cv_csv_path = os.path.join(output_dir, "5Fold_Cross_Validation_Summary.csv")
cv_df.to_csv(cv_csv_path, index=False)

print("\n" + "=" * 80)
print("5-FOLD CROSS-VALIDATION SUMMARY (MEAN ± STD)")
print("=" * 80)
for row in cv_summary:
    print(f"{row['Model']:<15} | "
          f"MAE: {row['CV MAE Mean']:.5f} ± {row['CV MAE Std']:.5f} | "
          f"RMSE: {row['CV RMSE Mean']:.5f} ± {row['CV RMSE Std']:.5f} | "
          f"R²: {row['CV R2 Mean']:.4f} ± {row['CV R2 Std']:.4f}")
print("=" * 80)

# ====================== MAE Bar Plot ======================
fig, ax = plt.subplots(figsize=(7.2, 5.2), facecolor="white")

models_list = list(models_cv.keys())
means = [cv_df.loc[cv_df["Model"] == m, "CV MAE Mean"].values[0] for m in models_list]
stds  = [cv_df.loc[cv_df["Model"] == m, "CV MAE Std"].values[0] for m in models_list]

x_pos = np.arange(len(models_list))
bars = ax.bar(x_pos, means, yerr=stds, capsize=5,
              color=[model_colors[m] for m in models_list],
              edgecolor="black", linewidth=0.9, alpha=0.88, width=0.55,
              error_kw=dict(ecolor="black", lw=1.1, capthick=1.1))

ymax = max(m + s for m, s in zip(means, stds))
ax.set_ylim(0, ymax * 1.25)
ax.set_ylabel("Mean Absolute Error (MAE)", fontsize=12)
ax.set_title("5-Fold Cross-Validation MAE Comparison", fontsize=13, fontweight="bold", pad=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(models_list, fontsize=11)
ax.tick_params(axis="y", labelsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.45)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for bar, mean, std in zip(bars, means, stds):
    height = bar.get_height()
    ax.annotate(f"{mean:.4f}\n±{std:.4f}",
                xy=(bar.get_x() + bar.get_width() / 2, height + std),
                xytext=(0, 7), textcoords="offset points",
                ha="center", va="bottom",
                fontsize=8.5, fontweight="bold", linespacing=1.1)

plt.tight_layout()
cv_plot_path = os.path.join(output_dir, "5Fold_CV_MAE_BarPlot.png")
plt.savefig(cv_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved: {cv_plot_path}")

# ====================== XGBoost Learning Curve (averaged over 5 folds) ======================
print("\nGenerating cross-validated XGBoost MAE Learning Curve...")

# Convert to arrays and average across folds
xgb_train_maes = np.array(xgb_train_maes)   # shape: (5, n_estimators)
xgb_val_maes   = np.array(xgb_val_maes)

mean_train = xgb_train_maes.mean(axis=0)
std_train  = xgb_train_maes.std(axis=0)
mean_val   = xgb_val_maes.mean(axis=0)
std_val    = xgb_val_maes.std(axis=0)

rounds = np.arange(1, len(mean_train) + 1)

fig, ax = plt.subplots(figsize=(7.5, 5.2), facecolor="white")

ax.plot(rounds, mean_train, label="Train MAE (mean)", color="#1f77b4", lw=2.2)
ax.fill_between(rounds, mean_train - std_train, mean_train + std_train,
                color="#1f77b4", alpha=0.15)

ax.plot(rounds, mean_val, label="Validation MAE (mean)", color="#ff7f0e", lw=2.2, linestyle="--")
ax.fill_between(rounds, mean_val - std_val, mean_val + std_val,
                color="#ff7f0e", alpha=0.15)

ax.set_title("XGBoost MAE Learning Curve (5-Fold CV)", fontweight="bold", fontsize=13, pad=12)
ax.set_xlabel("Number of Trees (Boosting Rounds)", fontsize=11)
ax.set_ylabel("Mean Absolute Error (MAE)", fontsize=11)
ax.legend(frameon=True, edgecolor="gray", fontsize=10)
ax.grid(True, linestyle="--", alpha=0.45)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
lc_plot_path = os.path.join(output_dir, "XGBoost_MAE_Learning_Curve.png")
plt.savefig(lc_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved: {lc_plot_path}")

# ====================== Final message ======================
print("\n" + "=" * 75)
print("EXECUTION COMPLETE – Files saved:")
print(" 1. 5Fold_Cross_Validation_Summary.csv")
print(" 2. 5Fold_CV_MAE_BarPlot.png")
print(" 3. XGBoost_MAE_Learning_Curve.png   ← now from the same 5-fold training")
print(f"\nAll files are in:\n{output_dir}")
print("=" * 75)

Reading dataset...
Dataset size: 7,270,392 rows

Running 5-Fold Cross-Validation...
 -> Training XGBoost on Fold 1/5...
 -> Training LightGBM on Fold 1/5...
 -> Training RandomForest on Fold 1/5...
 Fold 1/5 finished.

 -> Training XGBoost on Fold 2/5...
 -> Training LightGBM on Fold 2/5...
 -> Training RandomForest on Fold 2/5...
 Fold 2/5 finished.

 -> Training XGBoost on Fold 3/5...
 -> Training LightGBM on Fold 3/5...
 -> Training RandomForest on Fold 3/5...
 -> Training RandomForest on Fold 4/5...
 Fold 4/5 finished.

 -> Training XGBoost on Fold 5/5...
 -> Training RandomForest on Fold 5/5...


In [8]:
# ======================================================================
# SHAP + Permutation Importance Analysis of XGBoost
# High-quality Q1 journal style figures
# Year kept in training & importance calculation,
# but REMOVED from all final plots and summary tables
# ======================================================================

import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
import shap

warnings.filterwarnings("ignore")

# ======================================================================
# GLOBAL JOURNAL STYLE
# ======================================================================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
TEST_SIZE = 0.30

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.alpha": 0.35,
    "grid.linewidth": 0.6,
})

# Distinct colour palettes
PERM_COLOR = "#0D5C63"          # deep teal for permutation
SHAP_COLOR = "#3D348B"          # indigo for SHAP importance
SHAP_CMAP  = "coolwarm"         # for dependence plots

# ======================================================================
# PATHS
# ======================================================================
input_csv  = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\lagged_ml_ready.csv"
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\Shap_Analysis\Normalized dependence plot"
os.makedirs(output_dir, exist_ok=True)

# ======================================================================
# STEP 1: LOAD & SANITIZE DATA
# ======================================================================
print("Reading dataset...")
df = pd.read_csv(input_csv)
df.columns = df.columns.str.strip()
print(f"Dataset size: {len(df):,} rows")
print(f"All columns: {df.columns.tolist()}")

if "VCI" not in df.columns:
    raise ValueError("Target column 'VCI' not found!")

X = df.drop(columns=["VCI"])
y = df["VCI"].copy()
feature_names = X.columns.tolist()
print(f"\nFeatures used for training ({len(feature_names)}): {feature_names}")

# Normalize target only (features stay in original physical units)
target_scaler = MinMaxScaler()
y = target_scaler.fit_transform(y.values.reshape(-1, 1)).ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True
)
print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

# ======================================================================
# STEP 2: TRAIN XGBOOST (Year included)
# ======================================================================
print("\nTraining XGBoost Model (Year included)...")
model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.04,
    max_depth=15,
    min_child_weight=7,
    subsample=0.75,
    colsample_bytree=0.75,
    gamma=0.01,
    reg_alpha=0.8,
    reg_lambda=2.0,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
    eval_metric="rmse"
)
model.fit(X_train, y_train)
print("XGBoost training completed successfully.")

# ======================================================================
# STEP 3: PERMUTATION FEATURE IMPORTANCE (FULL test set)
# ======================================================================
print("\nCalculating Permutation Feature Importance on FULL test set...")
print(f"Using all {len(X_test):,} test samples...")

# n_jobs=1 avoids BrokenProcessPool on Windows
perm_result = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,                 # ← fixed
    scoring="r2"
)

perm_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance_Mean": perm_result.importances_mean,
    "Importance_Std":  perm_result.importances_std,
})

# Remove Year
perm_df = perm_df[~perm_df["Feature"].str.contains("year", case=False)].copy()

mean_imp = np.clip(perm_df["Importance_Mean"].values, 0, None)
total_imp = np.sum(mean_imp) if np.sum(mean_imp) > 0 else 1.0
perm_df["Importance_Relative_%"] = (mean_imp / total_imp) * 100
perm_df = perm_df.sort_values("Importance_Mean", ascending=True)

# Save CSV
perm_csv = os.path.join(output_dir, "XGBoost_Permutation_Importance_Summary.csv")
perm_df.sort_values("Importance_Mean", ascending=False).to_csv(perm_csv, index=False)
print(f"Saved summary: {perm_csv}")

# ---- Journal-style permutation bar plot ----
fig, ax = plt.subplots(figsize=(8.5, 6.5), facecolor="white")
bars = ax.barh(
    perm_df["Feature"],
    perm_df["Importance_Relative_%"],
    xerr=(perm_df["Importance_Std"] / total_imp) * 100,
    color=PERM_COLOR,
    alpha=0.88,
    edgecolor="black",
    linewidth=0.5,
    capsize=2.5,
    error_kw=dict(ecolor="#333333", lw=0.8)
)

ax.set_xlabel("Relative Permutation Importance Contribution (%)", fontweight="bold")
ax.set_title("XGBoost – Permutation Feature Importance (Full Test Set)",
             fontweight="bold", pad=10)
ax.set_xlim(0, perm_df["Importance_Relative_%"].max() * 1.18)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.6, bar.get_y() + bar.get_height()/2,
            f"{w:.1f}%", va="center", ha="left", fontsize=9,
            fontweight="medium", color=PERM_COLOR)

plt.tight_layout()
perm_plot = os.path.join(output_dir, "XGBoost_Permutation_Importance.png")
plt.savefig(perm_plot)
plt.close()
print(f"Saved plot : {perm_plot}")

# ======================================================================
# STEP 4: SHAP ANALYSIS (5 000 samples)
# ======================================================================
print("\nCalculating SHAP values (5,000 samples)...")
sample_size = min(5000, len(X_test))
X_shap = X_test.sample(n=sample_size, random_state=RANDOM_STATE)
print(f"SHAP sample size: {len(X_shap):,}")

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap)

# Identify & remove Year
year_mask = X_shap.columns.str.contains("year", case=False)
year_idx  = np.where(year_mask)[0]

if len(year_idx) > 0:
    print(f"Removing Year column(s) from SHAP results: {X_shap.columns[year_mask].tolist()}")
    shap_values = np.delete(shap_values, year_idx, axis=1)
    X_shap_no_year = X_shap.loc[:, ~year_mask]
    feature_names_no_year = X_shap_no_year.columns.tolist()
else:
    X_shap_no_year = X_shap
    feature_names_no_year = feature_names

# ----------------------------------------------------------------------
# 4.1 SHAP Summary (Beeswarm)
# ----------------------------------------------------------------------
print("Generating SHAP Summary Plot...")
plt.figure(figsize=(9.5, 7), facecolor="white")
shap.summary_plot(
    shap_values,
    X_shap_no_year,
    show=False,
    max_display=20,
    plot_size=None,
    cmap="RdBu_r"
)
plt.title("XGBoost – SHAP Summary Plot (Impact on Normalized VCI)",
          fontweight="bold", pad=12)
plt.xlabel("SHAP value (impact on model output)", fontweight="bold")
plt.tight_layout()
shap_summary_path = os.path.join(output_dir, "XGBoost_SHAP_Summary_Plot.png")
plt.savefig(shap_summary_path)
plt.close()
print(f"Saved plot : {shap_summary_path}")

# ----------------------------------------------------------------------
# 4.2 SHAP Feature Importance (Mean |SHAP|)
# ----------------------------------------------------------------------
print("Generating SHAP Feature Importance (Mean |SHAP|)...")
mean_abs_shap = np.abs(shap_values).mean(axis=0)
total_shap = np.sum(mean_abs_shap) if np.sum(mean_abs_shap) > 0 else 1.0
shap_relative_pct = (mean_abs_shap / total_shap) * 100

shap_imp_df = pd.DataFrame({
    "Feature": feature_names_no_year,
    "Mean_Abs_SHAP": mean_abs_shap,
    "Importance_Relative_%": shap_relative_pct
}).sort_values("Mean_Abs_SHAP", ascending=True)

shap_imp_csv = os.path.join(output_dir, "XGBoost_SHAP_Feature_Importance_Summary.csv")
shap_imp_df.sort_values("Mean_Abs_SHAP", ascending=False).to_csv(shap_imp_csv, index=False)
print(f"Saved summary: {shap_imp_csv}")

# Journal-style SHAP importance bar
fig, ax = plt.subplots(figsize=(8.5, 6.5), facecolor="white")
bars = ax.barh(
    shap_imp_df["Feature"],
    shap_imp_df["Importance_Relative_%"],
    color=SHAP_COLOR,
    alpha=0.88,
    edgecolor="black",
    linewidth=0.5
)

ax.set_xlabel("Relative SHAP Feature Importance Contribution (%)", fontweight="bold")
ax.set_title("XGBoost – SHAP Feature Importance (Mean Absolute SHAP)",
             fontweight="bold", pad=10)
ax.set_xlim(0, shap_imp_df["Importance_Relative_%"].max() * 1.18)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.6, bar.get_y() + bar.get_height()/2,
            f"{w:.1f}%", va="center", ha="left", fontsize=9,
            fontweight="medium", color=SHAP_COLOR)

plt.tight_layout()
shap_imp_plot = os.path.join(output_dir, "XGBoost_SHAP_Feature_Importance.png")
plt.savefig(shap_imp_plot)
plt.close()
print(f"Saved plot : {shap_imp_plot}")

# ----------------------------------------------------------------------
# 4.3 SHAP DEPENDENCE PLOTS – fully controlled journal style
# ----------------------------------------------------------------------
print("\nGenerating SHAP Dependence Plots (physical units + shared y-limits)...")

top_n = 6
top_features = (shap_imp_df
                .sort_values("Mean_Abs_SHAP", ascending=False)["Feature"]
                .head(top_n)
                .tolist())
print(f"Top {top_n} features for dependence plots: {top_features}")

# Shared y-limits
ymin = np.floor(shap_values.min() * 20) / 20
ymax = np.ceil(shap_values.max() * 20) / 20
ylim = (ymin, ymax)

# ---- Individual dependence plots ----
for feat in top_features:
    fig, ax = plt.subplots(figsize=(7.5, 5.8), facecolor="white")

    j = list(feature_names_no_year).index(feat)
    interaction = shap.utils.approximate_interactions(j, shap_values, X_shap_no_year)[0]
    color_feat = feature_names_no_year[interaction]

    sc = ax.scatter(
        X_shap_no_year[feat],
        shap_values[:, j],
        c=X_shap_no_year[color_feat],
        cmap=SHAP_CMAP,
        s=8,
        alpha=0.72,
        edgecolors="none",
        rasterized=True
    )

    ax.axhline(0, color="#555555", lw=0.9, ls="--", zorder=0)
    ax.set_xlabel(feat, fontweight="bold")
    ax.set_ylabel("SHAP value\n(change in normalized VCI)", fontweight="bold")
    ax.set_ylim(*ylim)
    ax.set_title(f"SHAP Dependence – {feat}", fontweight="bold", pad=8)

    cbar = fig.colorbar(sc, ax=ax, pad=0.02, aspect=30)
    cbar.set_label(color_feat, fontsize=10)
    cbar.ax.tick_params(labelsize=9)

    plt.tight_layout()
    dep_path = os.path.join(output_dir, f"XGBoost_SHAP_Dependence_{feat}.png")
    plt.savefig(dep_path)
    plt.close()
    print(f"  Saved: {dep_path}")

# ---- Combined 2×2 Top-4 figure ----
print("Generating combined dependence figure (Top-4)...")
top4 = top_features[:4]

fig, axes = plt.subplots(2, 2, figsize=(11.5, 9.2), facecolor="white")
axes = axes.ravel()

for i, feat in enumerate(top4):
    ax = axes[i]
    j = list(feature_names_no_year).index(feat)

    interaction = shap.utils.approximate_interactions(j, shap_values, X_shap_no_year)[0]
    color_feat = feature_names_no_year[interaction]

    sc = ax.scatter(
        X_shap_no_year[feat],
        shap_values[:, j],
        c=X_shap_no_year[color_feat],
        cmap=SHAP_CMAP,
        s=7,
        alpha=0.70,
        edgecolors="none",
        rasterized=True
    )

    ax.axhline(0, color="#555555", lw=0.85, ls="--", zorder=0)
    ax.set_xlabel(feat, fontweight="bold")
    ax.set_ylabel("SHAP value\n(change in normalized VCI)", fontweight="bold")
    ax.set_ylim(*ylim)
    ax.set_title(f"({chr(97+i)})  {feat}", loc="left", fontweight="bold", fontsize=12)

    cbar = fig.colorbar(sc, ax=ax, pad=0.015, aspect=28)
    cbar.set_label(color_feat, fontsize=9)
    cbar.ax.tick_params(labelsize=8)

fig.suptitle("XGBoost – SHAP Dependence Plots (Top-4 Features)",
             fontsize=14, fontweight="bold", y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.97])
combined_dep_path = os.path.join(output_dir, "XGBoost_SHAP_Dependence_Top4_Combined.png")
plt.savefig(combined_dep_path)
plt.close()
print(f"Saved combined: {combined_dep_path}")

# ======================================================================
# FINISHED
# ======================================================================
print("\n" + "=" * 72)
print("ALL DONE – Q1 journal style figures generated")
print("• Year included in model training & importance calculation")
print("• Year completely removed from every final plot & CSV")
print("• Permutation Importance → deep teal palette  (n_jobs=1)")
print("• SHAP Importance & Beeswarm → indigo palette")
print("• Dependence plots → original units on x, shared y-limits, coolwarm")
print("• All panels use consistent typography, spines, grid & DPI=300")
print("\nFiles generated:")
print(" 1. XGBoost_Permutation_Importance_Summary.csv")
print(" 2. XGBoost_Permutation_Importance.png")
print(" 3. XGBoost_SHAP_Summary_Plot.png")
print(" 4. XGBoost_SHAP_Feature_Importance_Summary.csv")
print(" 5. XGBoost_SHAP_Feature_Importance.png")
print(" 6. XGBoost_SHAP_Dependence_<Feature>.png  (individual)")
print(" 7. XGBoost_SHAP_Dependence_Top4_Combined.png")
print(f"\nSaved Location: {output_dir}")
print("=" * 72)

Reading dataset...
Dataset size: 7,270,392 rows
All columns: ['Year', 'Month_sin', 'Month_cos', 'LST', 'SMDI', 'SPEI_3', 'SPEI_6', 'LST_lag1', 'LST_lag2', 'SMDI_lag1', 'SMDI_lag2', 'SPEI_3_lag1', 'SPEI_3_lag2', 'SPEI_6_lag1', 'SPEI_6_lag2', 'VCI']

Features used for training (15): ['Year', 'Month_sin', 'Month_cos', 'LST', 'SMDI', 'SPEI_3', 'SPEI_6', 'LST_lag1', 'LST_lag2', 'SMDI_lag1', 'SMDI_lag2', 'SPEI_3_lag1', 'SPEI_3_lag2', 'SPEI_6_lag1', 'SPEI_6_lag2']
Training samples : 5,089,274
Testing samples  : 2,181,118

Training XGBoost Model (Year included)...
XGBoost training completed successfully.

Calculating Permutation Feature Importance on FULL test set...
Using all 2,181,118 test samples...
Saved summary: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\Shap_Analysis\Normalized dependence plot\XGBoost_Permutation_Importance_Summary.csv
Saved plot : E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\Shap_Analysis\Normalized dependence plot\XGBoost_Permuta

In [16]:
# ----------------------------------------------------------------------
# 4.3 SHAP DEPENDENCE PLOTS – CLEAN + LARGER POINTS
# ----------------------------------------------------------------------
print("\nGenerating clean SHAP Dependence Plots with larger points...")

top4 = ["LST", "SMDI", "SPEI_6", "SPEI_6_lag2"]

ymin = np.floor(shap_values.min() * 20) / 20
ymax = np.ceil(shap_values.max() * 20) / 20
ylim = (ymin, ymax)

fig, axes = plt.subplots(2, 2, figsize=(11, 9), facecolor="white")
axes = axes.ravel()

for i, feat in enumerate(top4):
    ax = axes[i]
    j = list(feature_names_no_year).index(feat)

    interaction = shap.utils.approximate_interactions(j, shap_values, X_shap_no_year)[0]
    color_feat = feature_names_no_year[interaction]

    sc = ax.scatter(
        X_shap_no_year[feat],
        shap_values[:, j],
        c=X_shap_no_year[color_feat],
        cmap="coolwarm",
        s=22,                # ← increased size (was 7)
        alpha=0.65,          # slightly lower alpha so it doesn’t look too dense
        edgecolors="none",
        rasterized=True
    )

    ax.axhline(0, color="grey", lw=0.8, ls="--", zorder=0)
    
    ax.set_xlabel(feat, fontweight="bold")
    ax.set_ylabel("SHAP value", fontweight="bold")
    ax.set_ylim(*ylim)
    
    ax.set_title(f"({chr(97+i)})", loc="left", fontweight="bold", fontsize=13, pad=6)

    cbar = fig.colorbar(sc, ax=ax, pad=0.02, aspect=30)
    cbar.set_label(color_feat, fontsize=9)
    cbar.ax.tick_params(labelsize=8)

plt.tight_layout()
combined_path = os.path.join(output_dir, "XGBoost_SHAP_Dep_Top4_Combined.png")
plt.savefig(combined_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved: {combined_path}")


Generating clean SHAP Dependence Plots with larger points...
Saved: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\Shap_Analysis\Normalized dependence plot\XGBoost_SHAP_Dep_Top4_Combined.png


In [ ]:
"""
Supplementary robustness checks a Q1 reviewer is likely to request.
Each block runs on outputs you already have; edit the paths/column names marked EDIT.
Report results alongside (not instead of) the existing random-split and pixel-wise results.
"""
import numpy as np, pandas as pd

# ---- 1. Spatial block and leave-one-year-out validation (same hyperparameters as the final XGBoost) ----
def blocked_validation(df, feature_cols, target="VCI_norm", model_params=None, cell_deg=0.25, n_splits=5):
    import xgboost as xgb
    from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
    from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
    params = model_params or dict(n_estimators=1000, learning_rate=0.04, max_depth=15, min_child_weight=7,
                                  subsample=0.75, colsample_bytree=0.75, gamma=0.01, reg_alpha=0.8, reg_lambda=2.0)
    cell = (np.floor(df["lat"] / cell_deg).astype(int).astype(str) + "_" + np.floor(df["lon"] / cell_deg).astype(int).astype(str))  # EDIT lat/lon columns
    out = {}
    for name, splitter, groups in (("spatial_block_5fold", GroupKFold(n_splits=n_splits), cell),
                                   ("leave_one_year_out", LeaveOneGroupOut(), df["year"])):          # EDIT year column
        scores = []
        for tr, te in splitter.split(df, groups=groups):
            m = xgb.XGBRegressor(**params, n_jobs=-1).fit(df.iloc[tr][feature_cols], df.iloc[tr][target])
            p = m.predict(df.iloc[te][feature_cols]); y = df.iloc[te][target]
            scores.append((r2_score(y, p), mean_squared_error(y, p) ** 0.5, mean_absolute_error(y, p)))
        out[name] = pd.DataFrame(scores, columns=["R2", "RMSE", "MAE"]).agg(["mean", "std"])
    return out   # report next to the random-split R2 = 0.84

# ---- 2. Collinearity: VIF for all 14 predictors and a Lag-0-only model ----
def vif_table(df, feature_cols):
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    X = df[feature_cols].dropna().sample(min(200_000, len(df)), random_state=42)
    return pd.Series([variance_inflation_factor(X.values, i) for i in range(X.shape[1])], index=feature_cols).sort_values(ascending=False)

# ---- 3. Pixel-wise trend significance: Benjamini-Hochberg FDR and a coarse-cell (0.25 deg) version ----
def fdr_significant_area(pvals, alpha=0.05):
    from statsmodels.stats.multitest import multipletests
    p = np.asarray(pvals, float); ok = np.isfinite(p)
    reject = np.zeros(p.shape, bool); reject[ok] = multipletests(p[ok], alpha=alpha, method="fdr_bh")[0]
    return reject.sum() / ok.sum() * 100          # compare with the uncorrected % (e.g. VCI 71.85)

def coarse_cell_trend(df_cell_year, value_col, year_col="year", cell_col="cell"):
    """Aggregate to 0.25 deg cells first, then run Mann-Kendall per cell (removes pseudo-replication for SMDI/SPEI)."""
    import pymannkendall as mk
    rows = []
    for c, g in df_cell_year.groupby(cell_col):
        r = mk.original_test(g.sort_values(year_col)[value_col].values)
        rows.append((c, r.slope, r.p))
    t = pd.DataFrame(rows, columns=["cell", "sen_slope", "p"])
    return t, (t.p < 0.05).mean() * 100

# ---- 4. VCI vs raw NDVI trend check (should agree in sign; VCI is an affine rescaling per pixel-month) ----
def ndvi_vs_vci_trend(ndvi_gs_mean_by_year, vci_gs_mean_by_year):
    from scipy import stats
    yrs = np.arange(2001, 2021)
    return {k: stats.linregress(yrs, np.asarray(v)) for k, v in (("NDVI", ndvi_gs_mean_by_year), ("VCI", vci_gs_mean_by_year))}

# ---- 5. Wider Random Forest search (comparable budget to XGBoost/LightGBM) ----
def rf_search(X_tr, y_tr, n_iter=30):
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import RandomizedSearchCV
    space = dict(n_estimators=[300, 500, 800], max_depth=[17, 25, 35, None], max_features=[0.33, 0.5, 0.8, 1.0],
                 min_samples_leaf=[1, 2, 5], min_samples_split=[2, 5, 10])
    return RandomizedSearchCV(RandomForestRegressor(n_jobs=-1, random_state=42), space, n_iter=n_iter, cv=5,
                              scoring="neg_root_mean_squared_error", random_state=42).fit(X_tr, y_tr)

In [1]:
# ============================================================
# COMPLETE ANALYSIS PIPELINE
# Updated RandomForest parameters + All figures + 5-Fold CV
# Output folder: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\ALL\Update
# ============================================================

import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import pearsonr, gaussian_kde
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

# ============================================================
# 1. CONFIGURATION
# ============================================================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
TEST_SIZE = 0.30

input_csv = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\lagged_ml_ready.csv"
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\ALL\Update"
os.makedirs(output_dir, exist_ok=True)

# Professional color palette
model_colors = {
    "XGBoost": "#1f77b4",       # Muted Blue
    "LightGBM": "#2ca02c",      # Muted Green
    "RandomForest": "#d62728"   # Muted Red
}

# Subfigure labels
labels_map = {
    "XGBoost": "a",
    "LightGBM": "b",
    "RandomForest": "c"
}

# ============================================================
# 2. LOAD & PREPROCESS DATA
# ============================================================
print("Reading dataset...")
df = pd.read_csv(input_csv)
print(f"Dataset size: {len(df):,} rows")

X = df.drop(columns=["VCI"])
y = df["VCI"].copy()

# Normalize target to [0, 1]
target_scaler = MinMaxScaler()
y = target_scaler.fit_transform(y.values.reshape(-1, 1)).ravel()

# 70/30 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

# ============================================================
# 3. DEFINE MODELS (UPDATED RandomForest)
# ============================================================
models = {
    "XGBoost": XGBRegressor(
        n_estimators=1000,
        learning_rate=0.04,
        max_depth=15,
        min_child_weight=7,
        subsample=0.75,
        colsample_bytree=0.75,
        gamma=0.01,
        reg_alpha=0.8,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
        eval_metric="rmse"
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=1400,
        learning_rate=0.04,
        max_depth=15,
        num_leaves=2048,
        min_child_samples=50,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.01,
        reg_lambda=0.2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    ),
    "RandomForest": RandomForestRegressor(
        n_estimators=800,               # UPDATED
        max_depth=18,                   # UPDATED
        min_samples_split=8,            # UPDATED
        min_samples_leaf=4,             # UPDATED
        max_features="sqrt",            # UPDATED
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

# ============================================================
# 4. TRAIN MODELS & COLLECT RESULTS
# ============================================================
model_results = {}
print("\nTraining models and collecting metrics...")

for name, model in models.items():
    print(f" → Training {name}...")
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred  = model.predict(X_test)

    train_r2   = r2_score(y_train, train_pred)
    test_r2    = r2_score(y_test, test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse  = np.sqrt(mean_squared_error(y_test, test_pred))
    train_mae  = mean_absolute_error(y_train, train_pred)
    test_mae   = mean_absolute_error(y_test, test_pred)
    pearson_r, _ = pearsonr(y_test, test_pred)

    model_results[name] = {
        "model": model,
        "train_pred": train_pred,
        "test_pred": test_pred,
        "train_r2": train_r2,
        "test_r2": test_r2,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "pearson_r": pearson_r,
        "train_res": y_train - train_pred,
        "test_res": y_test - test_pred
    }
    print(f"    Train R²: {train_r2:.4f} | Test R²: {test_r2:.4f}")

# Global plotting style
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["figure.dpi"] = 300
box_props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='lightgray')

# Sampling for scatter plots (max 100k points)
train_sample_size = min(100_000, len(y_train))
train_indices = np.random.RandomState(RANDOM_STATE).choice(len(y_train), size=train_sample_size, replace=False)
y_train_sample = y_train[train_indices]

test_sample_size = min(100_000, len(y_test))
test_indices = np.random.RandomState(RANDOM_STATE).choice(len(y_test), size=test_sample_size, replace=False)
y_test_sample = y_test[test_indices]

model_names = list(models.keys())

# ============================================================
# 5. TAYLOR DIAGRAM
# ============================================================
print("\nGenerating Taylor Diagram...")

std_ref = np.std(y_test)
model_stds = [np.std(res["test_pred"]) for res in model_results.values()]
max_std = max(model_stds + [std_ref]) * 1.25

fig = plt.figure(figsize=(8.5, 8), facecolor="white")
ax = fig.add_subplot(111, polar=True)
ax.set_thetamin(0)
ax.set_thetamax(90)
ax.set_ylim(0, max_std)

# Correlation labels
corr_labels = [0.0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99, 1.0]
theta_labels = np.degrees(np.arccos(corr_labels))
ax.set_thetagrids(theta_labels, labels=[f"{c:.2f}" for c in corr_labels], fontsize=10)
ax.set_ylabel("Standard Deviation", fontsize=12, labelpad=40)

# Correct RMSD contours
theta = np.linspace(0, np.pi/2, 400)
r = np.linspace(0, max_std, 400)
THETA, R = np.meshgrid(theta, r)
RMSD = np.sqrt(std_ref**2 + R**2 - 2 * std_ref * R * np.cos(THETA))
levels = np.linspace(0.05, max_std * 0.9, 7)
cs = ax.contour(THETA, R, RMSD, levels=levels, colors="gray",
                linestyles="--", linewidths=0.9, alpha=0.8)
ax.clabel(cs, inline=True, fontsize=8, fmt="%.2f")

# Observed reference
ax.plot(0, std_ref, marker="*", markersize=18, color="red",
        label="Observed VCI", zorder=10)

# Model points
for name, res in model_results.items():
    pred_std = np.std(res["test_pred"])
    r_val = res["pearson_r"]
    theta_model = np.arccos(np.clip(r_val, -1, 1))
    ax.plot(theta_model, pred_std,
            marker="o", markersize=11, linestyle="None",
            color=model_colors[name],
            label=f"{name}\nr = {r_val:.3f} | R² = {res['test_r2']:.3f}")

ax.set_title("Taylor Diagram – VCI Prediction Models", fontsize=14, fontweight="bold", pad=28)
ax.legend(loc="upper right", bbox_to_anchor=(1.42, 1.12), fontsize=9, frameon=True)
plt.tight_layout()

taylor_path = os.path.join(output_dir, "Taylor_Diagram_VCI_Models.png")
plt.savefig(taylor_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved: {taylor_path}")

# ============================================================
# 6. RESIDUAL ERROR DISTRIBUTION
# ============================================================
print("\nGenerating Residual Error Distribution plots...")

all_res = np.concatenate([res["test_res"] for res in model_results.values()])
x_min, x_max = all_res.min() - 0.05, all_res.max() + 0.05

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (name, res) in zip(axes, model_results.items()):
    color = model_colors[name]
    residuals = res["test_res"]

    ax.hist(residuals, bins=50, density=True, alpha=0.35, color=color,
            edgecolor="black", linewidth=0.5, label="Residuals")
    kde = stats.gaussian_kde(residuals)
    x_grid = np.linspace(x_min, x_max, 400)
    ax.plot(x_grid, kde(x_grid), color=color, linewidth=2.5, label="KDE")
    ax.axvline(0, color="black", linestyle="--", linewidth=1.2)

    mean_err = np.mean(residuals)
    std_err  = np.std(residuals)
    rmse     = np.sqrt(np.mean(residuals**2))

    ax.text(0.95, 0.95,
            f"Mean = {mean_err:.5f}\nStd = {std_err:.4f}\nRMSE = {rmse:.4f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.85, edgecolor="gray"))

    ax.set_xlim(x_min, x_max)
    ax.set_title(name, fontsize=13, fontweight="bold")
    ax.set_xlabel("Residual Error\n(Observed VCI − Predicted VCI)", fontsize=11)
    ax.grid(linestyle=":", alpha=0.5)

axes[0].set_ylabel("Frequency Density", fontsize=11)
fig.suptitle("Residual Error Distribution Comparison of Machine Learning Models",
             fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()

resid_path = os.path.join(output_dir, "All_Models_Residual_Distribution_Comparison.png")
plt.savefig(resid_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved: {resid_path}")

# ============================================================
# 7. INDIVIDUAL TRAIN vs TEST BOXPLOTS
# ============================================================
print("\nGenerating individual Train vs Test absolute error boxplots...")

for name, res in model_results.items():
    fig, ax = plt.subplots(figsize=(6, 5), facecolor="white")
    abs_train = np.abs(res["train_res"])
    abs_test  = np.abs(res["test_res"])

    bp = ax.boxplot([abs_train, abs_test],
                    labels=["Train Error", "Test Error"],
                    patch_artist=True, showfliers=False, widths=0.45,
                    medianprops=dict(color="black", linewidth=1.8))

    for patch in bp["boxes"]:
        patch.set_facecolor(model_colors[name])
        patch.set_alpha(0.75)
        patch.set_edgecolor("black")
        patch.set_linewidth(1.2)

    ax.set_title(f"{name} − Train vs Test Absolute Error", fontsize=11, fontweight="bold", pad=12)
    ax.set_ylabel("Absolute Residual Error (|Actual − Predicted|)", fontsize=10)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()

    path = os.path.join(output_dir, f"{name.replace(' ', '_')}_Train_vs_Test_Boxplot.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")

# ============================================================
# 8. COMPARATIVE TEST-SET BOXPLOT
# ============================================================
print("\nGenerating comparative test-set boxplot...")

fig, ax = plt.subplots(figsize=(8, 5.5), facecolor="white")
all_test_errors = [np.abs(model_results[m]["test_res"]) for m in model_names]

bp = ax.boxplot(all_test_errors, labels=model_names,
                patch_artist=True, showfliers=False, widths=0.45,
                medianprops=dict(color="black", linewidth=1.8))

for patch, name in zip(bp["boxes"], model_names):
    patch.set_facecolor(model_colors[name])
    patch.set_alpha(0.80)
    patch.set_edgecolor("black")
    patch.set_linewidth(1.2)

ax.set_title("Test Set Absolute Residual Error Comparison Across Models",
             fontsize=12, fontweight="bold", pad=12)
ax.set_ylabel("Absolute Residual Error (|Actual − Predicted|)", fontsize=11)
ax.grid(True, linestyle=":", alpha=0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()

comp_box_path = os.path.join(output_dir, "All_Models_Test_Boxplot_Comparison.png")
plt.savefig(comp_box_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved: {comp_box_path}")

# ============================================================
# 9. SCATTERPLOTS – DENSITY VERSION (original style)
# ============================================================
print("\nGenerating density-based scatterplots (individual + combined)...")

# 9A. Individual density scatterplots
for name in model_names:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor='white')
    tag = labels_map[name]

    # Training
    ax1 = axes[0]
    tr_preds_sample = model_results[name]["train_pred"][train_indices]
    xy_tr = np.vstack([y_train_sample, tr_preds_sample])
    density_tr = gaussian_kde(xy_tr)(xy_tr)
    idx_tr = density_tr.argsort()

    ax1.scatter(y_train_sample[idx_tr], tr_preds_sample[idx_tr],
                c=density_tr[idx_tr], cmap='Blues', s=4, alpha=0.6, linewidths=0)
    ax1.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)

    tr_text = (f"Train Metrics:\nR²: {model_results[name]['train_r2']:.4f}\n"
               f"RMSE: {model_results[name]['train_rmse']:.4f}\n"
               f"MAE: {model_results[name]['train_mae']:.4f}")
    ax1.text(0.04, 0.96, tr_text, transform=ax1.transAxes, fontsize=10,
             bbox=box_props, verticalalignment='top')

    ax1.set_xlim([-0.05, 1.05]); ax1.set_ylim([-0.05, 1.05])
    ax1.set_xlabel('Actual VCI', fontsize=11)
    ax1.set_ylabel('Predicted VCI', fontsize=11)
    ax1.set_title(f'({tag}1) {name} - Training Set', fontsize=12, fontweight='bold', pad=10)
    ax1.grid(True, linestyle=':', alpha=0.45)
    ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

    # Testing
    ax2 = axes[1]
    te_preds_sample = model_results[name]["test_pred"][test_indices]
    xy_te = np.vstack([y_test_sample, te_preds_sample])
    density_te = gaussian_kde(xy_te)(xy_te)
    idx_te = density_te.argsort()

    ax2.scatter(y_test_sample[idx_te], te_preds_sample[idx_te],
                c=density_te[idx_te], cmap='Blues', s=4, alpha=0.6, linewidths=0)
    ax2.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)

    te_text = (f"Test Metrics:\nR²: {model_results[name]['test_r2']:.4f}\n"
               f"RMSE: {model_results[name]['test_rmse']:.4f}\n"
               f"MAE: {model_results[name]['test_mae']:.4f}")
    ax2.text(0.04, 0.96, te_text, transform=ax2.transAxes, fontsize=10,
             bbox=box_props, verticalalignment='top')

    ax2.set_xlim([-0.05, 1.05]); ax2.set_ylim([-0.05, 1.05])
    ax2.set_xlabel('Actual VCI', fontsize=11)
    ax2.set_ylabel('Predicted VCI', fontsize=11)
    ax2.set_title(f'({tag}2) {name} - Test Set', fontsize=12, fontweight='bold', pad=10)
    ax2.grid(True, linestyle=':', alpha=0.45)
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

    plt.tight_layout()
    indiv_path = os.path.join(output_dir, f"{tag}_{name}_Density_Scatterplot.png")
    plt.savefig(indiv_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {indiv_path}")

# 9B. Combined 2×3 density grid
fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor='white')

# Row 0 – Training
for col_idx, name in enumerate(model_names):
    ax = axes[0, col_idx]
    tag = labels_map[name]
    preds_sample = model_results[name]["train_pred"][train_indices]

    xy = np.vstack([y_train_sample, preds_sample])
    density = gaussian_kde(xy)(xy)
    idx = density.argsort()

    ax.scatter(y_train_sample[idx], preds_sample[idx],
               c=density[idx], cmap='Blues', s=4, alpha=0.6, linewidths=0)
    ax.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)

    text_str = (f"Train Metrics:\nR²: {model_results[name]['train_r2']:.4f}\n"
                f"RMSE: {model_results[name]['train_rmse']:.4f}\n"
                f"MAE: {model_results[name]['train_mae']:.4f}")
    ax.text(0.04, 0.96, text_str, transform=ax.transAxes, fontsize=10,
            bbox=box_props, verticalalignment='top')

    ax.set_xlim([-0.05, 1.05]); ax.set_ylim([-0.05, 1.05])
    ax.set_ylabel('Predicted VCI', fontsize=11)
    ax.set_title(f'({tag}) {name} (Training)', fontsize=12, fontweight='bold', pad=10)
    ax.grid(True, linestyle=':', alpha=0.45)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Row 1 – Testing
for col_idx, name in enumerate(model_names):
    ax = axes[1, col_idx]
    tag = labels_map[name]
    preds_sample = model_results[name]["test_pred"][test_indices]

    xy = np.vstack([y_test_sample, preds_sample])
    density = gaussian_kde(xy)(xy)
    idx = density.argsort()

    ax.scatter(y_test_sample[idx], preds_sample[idx],
               c=density[idx], cmap='Blues', s=4, alpha=0.6, linewidths=0)
    ax.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)

    text_str = (f"Test Metrics:\nR²: {model_results[name]['test_r2']:.4f}\n"
                f"RMSE: {model_results[name]['test_rmse']:.4f}\n"
                f"MAE: {model_results[name]['test_mae']:.4f}")
    ax.text(0.04, 0.96, text_str, transform=ax.transAxes, fontsize=10,
            bbox=box_props, verticalalignment='top')

    ax.set_xlim([-0.05, 1.05]); ax.set_ylim([-0.05, 1.05])
    ax.set_xlabel('Actual VCI', fontsize=11)
    ax.set_ylabel('Predicted VCI', fontsize=11)
    ax.set_title(f'({tag}) {name} (Testing)', fontsize=12, fontweight='bold', pad=10)
    ax.grid(True, linestyle=':', alpha=0.45)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
combined_density_path = os.path.join(output_dir, "All_Models_Combined_Density_Grid_a_b_c.png")
plt.savefig(combined_density_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {combined_density_path}")

# ============================================================
# 10. SCATTERPLOTS – SOLID DODGERBLUE VERSION
# ============================================================
print("\nGenerating solid-color (dodgerblue) scatterplots...")

# 10A. Individual solid scatterplots
for name in model_names:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor='white')

    # Training
    ax1 = axes[0]
    tr_preds_sample = model_results[name]["train_pred"][train_indices]
    ax1.scatter(y_train_sample, tr_preds_sample, color='dodgerblue', alpha=0.25, s=4, linewidths=0)
    ax1.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)

    tr_text = (f"Train Metrics:\nR²: {model_results[name]['train_r2']:.4f}\n"
               f"RMSE: {model_results[name]['train_rmse']:.4f}\n"
               f"MAE: {model_results[name]['train_mae']:.4f}")
    ax1.text(0.04, 0.96, tr_text, transform=ax1.transAxes, fontsize=10,
             bbox=box_props, verticalalignment='top')

    ax1.set_xlim([-0.05, 1.05]); ax1.set_ylim([-0.05, 1.05])
    ax1.set_xlabel('Actual VCI', fontsize=11)
    ax1.set_ylabel('Predicted VCI', fontsize=11)
    ax1.set_title(f'{name} - Training Set', fontsize=12, fontweight='bold', pad=10)
    ax1.grid(True, linestyle=':', alpha=0.45)
    ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

    # Testing
    ax2 = axes[1]
    te_preds_sample = model_results[name]["test_pred"][test_indices]
    ax2.scatter(y_test_sample, te_preds_sample, color='dodgerblue', alpha=0.25, s=4, linewidths=0)
    ax2.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)

    te_text = (f"Test Metrics:\nR²: {model_results[name]['test_r2']:.4f}\n"
               f"RMSE: {model_results[name]['test_rmse']:.4f}\n"
               f"MAE: {model_results[name]['test_mae']:.4f}")
    ax2.text(0.04, 0.96, te_text, transform=ax2.transAxes, fontsize=10,
             bbox=box_props, verticalalignment='top')

    ax2.set_xlim([-0.05, 1.05]); ax2.set_ylim([-0.05, 1.05])
    ax2.set_xlabel('Actual VCI', fontsize=11)
    ax2.set_ylabel('Predicted VCI', fontsize=11)
    ax2.set_title(f'{name} - Testing Set', fontsize=12, fontweight='bold', pad=10)
    ax2.grid(True, linestyle=':', alpha=0.45)
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

    plt.tight_layout()
    indiv_path = os.path.join(output_dir, f"{name}_Train_Test_Scatterplot_Solid.png")
    plt.savefig(indiv_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {indiv_path}")

# 10B. Combined solid 2×3 grid
fig, axes = plt.subplots(2, 3, figsize=(16, 10), facecolor='white')

# Row 0 – Training
for col_idx, name in enumerate(model_names):
    ax = axes[0, col_idx]
    tr_preds_sample = model_results[name]["train_pred"][train_indices]
    ax.scatter(y_train_sample, tr_preds_sample, color='dodgerblue', alpha=0.25, s=4, linewidths=0)
    ax.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)

    text_str = (f"Train Metrics:\nR²: {model_results[name]['train_r2']:.4f}\n"
                f"RMSE: {model_results[name]['train_rmse']:.4f}\n"
                f"MAE: {model_results[name]['train_mae']:.4f}")
    ax.text(0.04, 0.96, text_str, transform=ax.transAxes, fontsize=10,
            bbox=box_props, verticalalignment='top')

    ax.set_xlim([-0.05, 1.05]); ax.set_ylim([-0.05, 1.05])
    ax.set_ylabel('Predicted VCI', fontsize=11)
    ax.set_title(f'{name} (Training)', fontsize=12, fontweight='bold', pad=10)
    ax.grid(True, linestyle=':', alpha=0.45)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Row 1 – Testing
for col_idx, name in enumerate(model_names):
    ax = axes[1, col_idx]
    te_preds_sample = model_results[name]["test_pred"][test_indices]
    ax.scatter(y_test_sample, te_preds_sample, color='dodgerblue', alpha=0.25, s=4, linewidths=0)
    ax.plot([0, 1], [0, 1], color='red', linestyle='--', lw=1.5)

    text_str = (f"Test Metrics:\nR²: {model_results[name]['test_r2']:.4f}\n"
                f"RMSE: {model_results[name]['test_rmse']:.4f}\n"
                f"MAE: {model_results[name]['test_mae']:.4f}")
    ax.text(0.04, 0.96, text_str, transform=ax.transAxes, fontsize=10,
            bbox=box_props, verticalalignment='top')

    ax.set_xlim([-0.05, 1.05]); ax.set_ylim([-0.05, 1.05])
    ax.set_xlabel('Actual VCI', fontsize=11)
    ax.set_ylabel('Predicted VCI', fontsize=11)
    ax.set_title(f'{name} (Testing)', fontsize=12, fontweight='bold', pad=10)
    ax.grid(True, linestyle=':', alpha=0.45)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
grid_path = os.path.join(output_dir, "All_Models_Combined_Train_Test_Grid_Solid.png")
plt.savefig(grid_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: {grid_path}")

# ============================================================
# 11. 5-FOLD CROSS-VALIDATION (NO Learning Curve)
# ============================================================
print("\n" + "="*80)
print("Starting 5-Fold Cross-Validation...")
print("="*80)

# Convert to numpy for CV
X_np = X.to_numpy()
y_np = y

models_cv = {
    "XGBoost": lambda: XGBRegressor(
        n_estimators=1000,
        learning_rate=0.04,
        max_depth=15,
        min_child_weight=7,
        subsample=0.75,
        colsample_bytree=0.75,
        gamma=0.01,
        reg_alpha=0.8,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    ),
    "LightGBM": lambda: LGBMRegressor(
        n_estimators=1400,
        learning_rate=0.04,
        max_depth=15,
        num_leaves=2048,
        min_child_samples=50,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.01,
        reg_lambda=0.2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    ),
    "RandomForest": lambda: RandomForestRegressor(
        n_estimators=800,               # UPDATED
        max_depth=18,                   # UPDATED
        min_samples_split=8,            # UPDATED
        min_samples_leaf=4,             # UPDATED
        max_features="sqrt",            # UPDATED
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {m: {"val_mae": [], "val_rmse": [], "val_r2": []} for m in models_cv}

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np), 1):
    X_tr, X_val = X_np[train_idx], X_np[val_idx]
    y_tr, y_val = y_np[train_idx], y_np[val_idx]

    for name, model_fn in models_cv.items():
        print(f" → Training {name} on Fold {fold}/5...")
        model = model_fn()
        model.fit(X_tr, y_tr)
        val_pred = model.predict(X_val)

        cv_results[name]["val_mae"].append(mean_absolute_error(y_val, val_pred))
        cv_results[name]["val_rmse"].append(np.sqrt(mean_squared_error(y_val, val_pred)))
        cv_results[name]["val_r2"].append(r2_score(y_val, val_pred))

    print(f" Fold {fold}/5 finished.\n")

# Summary table
cv_summary = []
for name in models_cv:
    mae  = np.array(cv_results[name]["val_mae"])
    rmse = np.array(cv_results[name]["val_rmse"])
    r2   = np.array(cv_results[name]["val_r2"])
    cv_summary.append({
        "Model": name,
        "CV MAE Mean": np.mean(mae),
        "CV MAE Std": np.std(mae),
        "CV RMSE Mean": np.mean(rmse),
        "CV RMSE Std": np.std(rmse),
        "CV R2 Mean": np.mean(r2),
        "CV R2 Std": np.std(r2)
    })

cv_df = pd.DataFrame(cv_summary)
cv_csv_path = os.path.join(output_dir, "5Fold_Cross_Validation_Summary.csv")
cv_df.to_csv(cv_csv_path, index=False)

print("\n" + "="*80)
print("5-FOLD CROSS-VALIDATION SUMMARY (MEAN ± STD)")
print("="*80)
for row in cv_summary:
    print(f"{row['Model']:<15} | "
          f"MAE: {row['CV MAE Mean']:.5f} ± {row['CV MAE Std']:.5f} | "
          f"RMSE: {row['CV RMSE Mean']:.5f} ± {row['CV RMSE Std']:.5f} | "
          f"R²: {row['CV R2 Mean']:.4f} ± {row['CV R2 Std']:.4f}")
print("="*80)

# MAE Bar Plot
fig, ax = plt.subplots(figsize=(7.2, 5.2), facecolor="white")
models_list = list(models_cv.keys())
means = [cv_df.loc[cv_df["Model"] == m, "CV MAE Mean"].values[0] for m in models_list]
stds  = [cv_df.loc[cv_df["Model"] == m, "CV MAE Std"].values[0] for m in models_list]
x_pos = np.arange(len(models_list))

bars = ax.bar(x_pos, means, yerr=stds, capsize=5,
              color=[model_colors[m] for m in models_list],
              edgecolor="black", linewidth=0.9, alpha=0.88, width=0.55,
              error_kw=dict(ecolor="black", lw=1.1, capthick=1.1))

ymax = max(m + s for m, s in zip(means, stds))
ax.set_ylim(0, ymax * 1.25)
ax.set_ylabel("Mean Absolute Error (MAE)", fontsize=12)
ax.set_title("5-Fold Cross-Validation MAE Comparison", fontsize=13, fontweight="bold", pad=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(models_list, fontsize=11)
ax.tick_params(axis="y", labelsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.45)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for bar, mean, std in zip(bars, means, stds):
    height = bar.get_height()
    ax.annotate(f"{mean:.4f}\n±{std:.4f}",
                xy=(bar.get_x() + bar.get_width() / 2, height + std),
                xytext=(0, 7), textcoords="offset points",
                ha="center", va="bottom",
                fontsize=8.5, fontweight="bold", linespacing=1.1)

plt.tight_layout()
cv_plot_path = os.path.join(output_dir, "5Fold_CV_MAE_BarPlot.png")
plt.savefig(cv_plot_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved: {cv_plot_path}")

# ============================================================
# FINAL MESSAGE
# ============================================================
print("\n" + "="*75)
print("ALL ANALYSIS COMPLETED SUCCESSFULLY")
print("="*75)
print(f"All figures and results saved to:\n{output_dir}")
print("\nFiles generated:")
print("  • Taylor_Diagram_VCI_Models.png")
print("  • All_Models_Residual_Distribution_Comparison.png")
print("  • XGBoost / LightGBM / RandomForest_Train_vs_Test_Boxplot.png")
print("  • All_Models_Test_Boxplot_Comparison.png")
print("  • Density scatterplots (individual + combined grid)")
print("  • Solid dodgerblue scatterplots (individual + combined grid)")
print("  • 5Fold_Cross_Validation_Summary.csv")
print("  • 5Fold_CV_MAE_BarPlot.png")
print("="*75)

Reading dataset...
Dataset size: 7,270,392 rows
Training samples : 5,089,274
Testing samples  : 2,181,118

Training models and collecting metrics...
 → Training XGBoost...
    Train R²: 0.8743 | Test R²: 0.8403
 → Training LightGBM...
    Train R²: 0.8528 | Test R²: 0.8272
 → Training RandomForest...
    Train R²: 0.7876 | Test R²: 0.7676

Generating Taylor Diagram...
Saved: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\ALL\Update\Taylor_Diagram_VCI_Models.png

Generating Residual Error Distribution plots...
Saved: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\ALL\Update\All_Models_Residual_Distribution_Comparison.png

Generating individual Train vs Test absolute error boxplots...
Saved: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\ALL\Update\XGBoost_Train_vs_Test_Boxplot.png
Saved: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\ALL\Update\LightGBM_Train_vs_Test_Boxplot.png
Saved: E:\2_Contribution\Final\Preprocess

In [5]:
# ======================================================================
# Leave-One-Year-Out (LOYO) Cross-Validation of XGBoost
# Tests temporal generalization: train on 19 years, test on 1 unseen year
# Includes R2, RMSE, MAE, Pearson r, Spearman rho, and mean bias
# Q1 journal style figures
# ======================================================================

import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# ======================================================================
# GLOBAL JOURNAL STYLE
# ======================================================================
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.alpha": 0.35,
    "grid.linewidth": 0.6,
})

LOYO_COLOR = "#B23A48"
CORR_COLOR = "#1F6E8C"

# ======================================================================
# PATHS
# ======================================================================
input_csv  = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\Spearman\lagged_ml_ready.csv"
output_dir = r"E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\LOYO_Validation"
os.makedirs(output_dir, exist_ok=True)

# ======================================================================
# STEP 1: LOAD & SANITIZE DATA
# ======================================================================
print("Reading dataset...")
df = pd.read_csv(input_csv)
df.columns = df.columns.str.strip()
print(f"Dataset size: {len(df):,} rows")

if "VCI" not in df.columns:
    raise ValueError("Target column 'VCI' not found!")

year_col_matches = [c for c in df.columns if "year" in c.lower()]
if len(year_col_matches) == 0:
    raise ValueError("No 'Year' column found in dataset — required for LOYO splitting!")
year_col = year_col_matches[0]
print(f"Using '{year_col}' as the temporal grouping variable for LOYO CV.")

years = sorted(df[year_col].unique())
print(f"Years found ({len(years)}): {years}")

y_all = df["VCI"].copy()
target_scaler = MinMaxScaler()
y_all_norm = target_scaler.fit_transform(y_all.values.reshape(-1, 1)).ravel()
df["_VCI_norm"] = y_all_norm

feature_cols = [c for c in df.columns if c not in ["VCI", "_VCI_norm", year_col]]
print(f"\nFeatures used for training ({len(feature_cols)}): {feature_cols}")

# ======================================================================
# STEP 2: LEAVE-ONE-YEAR-OUT CROSS-VALIDATION
# ======================================================================
print("\nRunning Leave-One-Year-Out Cross-Validation...")
print("(Same fixed XGBoost hyperparameters as the final optimized model — no re-tuning per fold)\n")

results = []

for test_year in years:
    train_df = df[df[year_col] != test_year]
    test_df  = df[df[year_col] == test_year]

    X_train = train_df[feature_cols]
    y_train = train_df["_VCI_norm"].values

    X_test  = test_df[feature_cols]
    y_test  = test_df["_VCI_norm"].values

    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.04,
        max_depth=15,
        min_child_weight=7,
        subsample=0.75,
        colsample_bytree=0.75,
        gamma=0.01,
        reg_alpha=0.8,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
        eval_metric="rmse"
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    pearson_r, _  = pearsonr(y_test, y_pred)
    spearman_r, _ = spearmanr(y_test, y_pred)
    bias = np.mean(y_pred - y_test)

    results.append({
        "Held_Out_Year": test_year,
        "N_Test_Samples": len(y_test),
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "Pearson_r": pearson_r,
        "Spearman_rho": spearman_r,
        "Mean_Bias": bias
    })

    print(f"Year {test_year}: n={len(y_test):>7,}  R²={r2:7.4f}  RMSE={rmse:.4f}  "
          f"MAE={mae:.4f}  Pearson_r={pearson_r:.3f}  Spearman={spearman_r:.3f}  Bias={bias:+.4f}")

# ======================================================================
# STEP 3: SUMMARIZE RESULTS
# ======================================================================
loyo_df = pd.DataFrame(results)

summary_row = pd.DataFrame([{
    "Held_Out_Year": "Mean ± SD",
    "N_Test_Samples": int(loyo_df["N_Test_Samples"].mean()),
    "R2": loyo_df["R2"].mean(),
    "RMSE": loyo_df["RMSE"].mean(),
    "MAE": loyo_df["MAE"].mean(),
    "Pearson_r": loyo_df["Pearson_r"].mean(),
    "Spearman_rho": loyo_df["Spearman_rho"].mean(),
    "Mean_Bias": loyo_df["Mean_Bias"].mean()
}])

loyo_df_with_summary = pd.concat([loyo_df, summary_row], ignore_index=True)

print("\n" + "=" * 90)
print("LOYO CROSS-VALIDATION SUMMARY")
print(f"Mean R²        : {loyo_df['R2'].mean():.4f} ± {loyo_df['R2'].std():.4f}")
print(f"Mean RMSE      : {loyo_df['RMSE'].mean():.4f} ± {loyo_df['RMSE'].std():.4f}")
print(f"Mean MAE       : {loyo_df['MAE'].mean():.4f} ± {loyo_df['MAE'].std():.4f}")
print(f"Mean Pearson r : {loyo_df['Pearson_r'].mean():.4f} ± {loyo_df['Pearson_r'].std():.4f}")
print(f"Mean Spearman ρ: {loyo_df['Spearman_rho'].mean():.4f} ± {loyo_df['Spearman_rho'].std():.4f}")
print(f"Mean Bias      : {loyo_df['Mean_Bias'].mean():+.4f} ± {loyo_df['Mean_Bias'].std():.4f}")
print("=" * 90)

loyo_csv = os.path.join(output_dir, "XGBoost_LOYO_CV_Results.csv")
loyo_df_with_summary.to_csv(loyo_csv, index=False)
print(f"\nSaved results: {loyo_csv}")

# ======================================================================
# STEP 4: FIGURE — R² PER HELD-OUT YEAR
# ======================================================================
print("\nGenerating LOYO R² bar chart...")

fig, ax = plt.subplots(figsize=(11, 6), facecolor="white")
mean_r2 = loyo_df["R2"].mean()
colors = [LOYO_COLOR if r2 >= mean_r2 else "#D9A6AD" for r2 in loyo_df["R2"]]

bars = ax.bar(
    loyo_df["Held_Out_Year"].astype(str),
    loyo_df["R2"],
    color=colors,
    edgecolor="black",
    linewidth=0.6,
    alpha=0.9,
    width=0.65
)

ax.axhline(0, color="#999999", lw=1.0, ls="-", zorder=0)
ax.axhline(mean_r2, color="#333333", lw=1.2, ls="--",
           label=f"Mean R² = {mean_r2:.3f} ± {loyo_df['R2'].std():.3f}")

ax.set_xlabel("Held-Out Year", fontweight="bold")
ax.set_ylabel("R² (Held-Out Year)", fontweight="bold")
ax.set_title("XGBoost – Leave-One-Year-Out Cross-Validation Performance",
             fontweight="bold", pad=10)
plt.xticks(rotation=45, ha="right")
ax.legend(loc="lower right", frameon=False)

for bar, r2 in zip(bars, loyo_df["R2"]):
    va = "bottom" if r2 >= 0 else "top"
    offset = 0.02 if r2 >= 0 else -0.02
    ax.text(bar.get_x() + bar.get_width()/2, r2 + offset,
            f"{r2:.2f}", ha="center", va=va, fontsize=8)

plt.tight_layout()
loyo_plot = os.path.join(output_dir, "XGBoost_LOYO_CV_R2_BarChart.png")
plt.savefig(loyo_plot)
plt.close()
print(f"Saved plot : {loyo_plot}")

# ======================================================================
# STEP 5: FIGURE — PEARSON / SPEARMAN CORRELATION PER HELD-OUT YEAR
# ======================================================================
print("Generating LOYO correlation (Pearson/Spearman) line chart...")

fig, ax = plt.subplots(figsize=(11, 6), facecolor="white")

ax.plot(loyo_df["Held_Out_Year"].astype(str), loyo_df["Pearson_r"],
        marker="o", color=CORR_COLOR, lw=1.8, label="Pearson r")
ax.plot(loyo_df["Held_Out_Year"].astype(str), loyo_df["Spearman_rho"],
        marker="s", color="#E08E45", lw=1.8, ls="--", label="Spearman ρ")

ax.axhline(0, color="#999999", lw=1.0, zorder=0)
ax.set_xlabel("Held-Out Year", fontweight="bold")
ax.set_ylabel("Correlation (Predicted vs. Observed VCI)", fontweight="bold")
ax.set_title("XGBoost – LOYO Predicted–Observed Correlation Retained per Year",
             fontweight="bold", pad=10)
plt.xticks(rotation=45, ha="right")
ax.legend(loc="lower right", frameon=False)

plt.tight_layout()
corr_plot = os.path.join(output_dir, "XGBoost_LOYO_CV_Correlation_LineChart.png")
plt.savefig(corr_plot)
plt.close()
print(f"Saved plot : {corr_plot}")

# ======================================================================
# STEP 6: FIGURE — MAE PER HELD-OUT YEAR
# ======================================================================
print("Generating LOYO MAE bar chart...")

fig, ax = plt.subplots(figsize=(11, 6), facecolor="white")
mean_mae = loyo_df["MAE"].mean()

ax.bar(
    loyo_df["Held_Out_Year"].astype(str),
    loyo_df["MAE"],
    color=LOYO_COLOR,
    edgecolor="black",
    linewidth=0.6,
    alpha=0.85,
    width=0.65
)

ax.axhline(mean_mae, color="#333333", lw=1.2, ls="--",
           label=f"Mean MAE = {mean_mae:.4f} ± {loyo_df['MAE'].std():.4f}")

ax.set_xlabel("Held-Out Year", fontweight="bold")
ax.set_ylabel("MAE (normalized VCI, Held-Out Year)", fontweight="bold")
ax.set_title("XGBoost – Leave-One-Year-Out Cross-Validation (MAE)",
             fontweight="bold", pad=10)
plt.xticks(rotation=45, ha="right")
ax.legend(loc="upper right", frameon=False)

plt.tight_layout()
loyo_mae_plot = os.path.join(output_dir, "XGBoost_LOYO_CV_MAE_BarChart.png")
plt.savefig(loyo_mae_plot)
plt.close()
print(f"Saved plot : {loyo_mae_plot}")

# ======================================================================
# FINISHED
# ======================================================================
print("\n" + "=" * 90)
print("ALL DONE – Leave-One-Year-Out Cross-Validation Completed")
print("Files generated:")
print(" 1. XGBoost_LOYO_CV_Results.csv              (per-year + mean±SD: R2, RMSE, MAE, Pearson, Spearman, Bias)")
print(" 2. XGBoost_LOYO_CV_R2_BarChart.png")
print(" 3. XGBoost_LOYO_CV_Correlation_LineChart.png")
print(" 4. XGBoost_LOYO_CV_MAE_BarChart.png")
print(f"\nSaved Location: {output_dir}")
print("=" * 90)

Reading dataset...
Dataset size: 7,270,392 rows
Using 'Year' as the temporal grouping variable for LOYO CV.
Years found (20): [np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]

Features used for training (14): ['Month_sin', 'Month_cos', 'LST', 'SMDI', 'SPEI_3', 'SPEI_6', 'LST_lag1', 'LST_lag2', 'SMDI_lag1', 'SMDI_lag2', 'SPEI_3_lag1', 'SPEI_3_lag2', 'SPEI_6_lag1', 'SPEI_6_lag2']

Running Leave-One-Year-Out Cross-Validation...
(Same fixed XGBoost hyperparameters as the final optimized model — no re-tuning per fold)

Year 2002: n=368,788  R²=-0.1671  RMSE=0.2954  MAE=0.2445  Pearson_r=0.143  Spearman=0.120  Bias=+0.1084
Year 2003: n=368,788  R²=-0.1252  RMSE=0.2357  MAE=0.1941  Pearson_r=0.101  Spearman=0.100  Bias=+0.071

In [8]:
# ======================================================================
# STEP 7: TABLE 4.4 VCI SEVERITY MERGE + COMBINED FIGURE (R² + Bias-vs-Severity)
# ======================================================================
print("\nGenerating combined Figure: Panel (a) R² per year, Panel (b) Bias vs. VCI severity...")

# Option A: if you have Table 4.4 as a CSV with columns ["Year", "Mean_VCI"]
# table_4_4_df = pd.read_csv(r"path\to\table_4_4.csv")

# Option B: quick manual dict if you don't have it as a file (edit values to match your actual Table 4.4)
table_4_4_dict = {
    # "Year": Mean_VCI   <-- fill in all 20 years from your Table 4.4
    2001: 16.69,
    2005: 27.43,
    2006: 26.34,
    # ... add remaining 17 years here
}
table_4_4_df = pd.DataFrame(list(table_4_4_dict.items()), columns=["Year", "Mean_VCI"])

# Merge with LOYO results
merged = loyo_df.merge(table_4_4_df, left_on="Held_Out_Year", right_on="Year", how="inner")

drought_years = [2001, 2005, 2006]
merged["highlight"] = merged["Held_Out_Year"].isin(drought_years)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor="white")

# ---- Panel (a): R² per held-out year (reuse existing logic) ----
ax = axes[0]
mean_r2 = loyo_df["R2"].mean()
colors = [LOYO_COLOR if r2 >= mean_r2 else "#D9A6AD" for r2 in loyo_df["R2"]]
ax.bar(loyo_df["Held_Out_Year"].astype(str), loyo_df["R2"],
       color=colors, edgecolor="black", linewidth=0.6, alpha=0.9, width=0.65)
ax.axhline(0, color="#999999", lw=1.0)
ax.axhline(mean_r2, color="#333333", lw=1.2, ls="--",
           label=f"Mean R² = {mean_r2:.3f}")
ax.set_xlabel("Held-Out Year", fontweight="bold")
ax.set_ylabel("R² (Held-Out Year)", fontweight="bold")
ax.set_title("(a) LOYO R² per Held-Out Year", fontweight="bold")
ax.tick_params(axis='x', rotation=45)
ax.legend(loc="lower right", frameon=False)

# ---- Panel (b): Bias vs. VCI severity scatter ----
ax = axes[1]
non_drought = merged[~merged["highlight"]]
drought = merged[merged["highlight"]]

ax.scatter(non_drought["Mean_VCI"], non_drought["Mean_Bias"],
           color="#A9A9A9", edgecolor="black", linewidth=0.5, s=60,
           label="Other years", zorder=2)
ax.scatter(drought["Mean_VCI"], drought["Mean_Bias"],
           color=LOYO_COLOR, edgecolor="black", linewidth=0.8, s=110,
           marker="D", label="Representative drought years", zorder=3)

for _, row in drought.iterrows():
    ax.annotate(str(int(row["Held_Out_Year"])),
                (row["Mean_VCI"], row["Mean_Bias"]),
                textcoords="offset points", xytext=(6, 6), fontsize=9, fontweight="bold")

ax.axhline(0, color="#999999", lw=1.0, ls="-")
ax.set_xlabel("Mean VCI (Severity, Table 4.4)", fontweight="bold")
ax.set_ylabel("Mean Bias (Predicted − Observed VCI)", fontweight="bold")
ax.set_title("(b) Bias vs. Drought Severity", fontweight="bold")
ax.legend(loc="upper right", frameon=False)

plt.tight_layout()
combined_plot = os.path.join(output_dir, "XGBoost_LOYO_Combined_R2_BiasSeverity.png")
plt.savefig(combined_plot)
plt.close()
print(f"Saved combined figure: {combined_plot}")


Generating combined Figure: Panel (a) R² per year, Panel (b) Bias vs. VCI severity...
Saved combined figure: E:\2_Contribution\Final\Preprocess\1_ML\Pre_Processed\ALL model\LOYO_Validation\XGBoost_LOYO_Combined_R2_BiasSeverity.png


In [9]:
# ======================================================================
# STEP 7: LOAD TABLE 4.4 (VCI SEVERITY PER YEAR) AND MERGE WITH LOYO RESULTS
# ======================================================================
print("\n" + "="*90)
print("STEP 7: Merging LOYO results with Table 4.4 VCI severity values")
print("="*90)

# ----------------------------------------------------------------------
# OPTION A (RECOMMENDED): If you already have Table 4.4 saved as a CSV
# somewhere in your pipeline, point to it here. It must contain at least
# two columns: a Year column and a Mean VCI column.
# ----------------------------------------------------------------------
table_4_4_csv_path = r"E:\2_Contribution\Final\Preprocess\3. Trend\Update result\Table_4_4_recomputed.csv"

if os.path.exists(table_4_4_csv_path):
    table_4_4_df = pd.read_csv(table_4_4_csv_path)
    table_4_4_df.columns = table_4_4_df.columns.str.strip()
    # Standardize column names if needed — EDIT these if your file uses different headers
    rename_map = {}
    for c in table_4_4_df.columns:
        if "year" in c.lower():
            rename_map[c] = "Year"
        if "vci" in c.lower() and "mean" in c.lower():
            rename_map[c] = "Mean_VCI"
    table_4_4_df = table_4_4_df.rename(columns=rename_map)
    print(f"Loaded Table 4.4 from file: {table_4_4_csv_path}")
else:
    # ----------------------------------------------------------------------
    # OPTION B (FALLBACK): Manually type in all 20 years' Mean VCI values
    # from your Table 4.4 as written in your thesis. REPLACE the ___ below
    # with your actual numbers for every year.
    # ----------------------------------------------------------------------
    print(f"File not found at {table_4_4_csv_path} — using manual dictionary instead.")
    print("!! EDIT the dictionary below with your real Table 4.4 values before trusting the figure !!")

    table_4_4_dict = {
        2001: 16.69,
        2002: None,   # <-- fill in
        2003: None,   # <-- fill in
        2004: None,   # <-- fill in
        2005: 27.43,
        2006: 26.34,
        2007: None,   # <-- fill in
        2008: None,   # <-- fill in
        2009: None,   # <-- fill in
        2010: None,   # <-- fill in
        2011: None,   # <-- fill in
        2012: None,   # <-- fill in
        2013: None,   # <-- fill in
        2014: None,   # <-- fill in
        2015: None,   # <-- fill in
        2016: None,   # <-- fill in
        2017: None,   # <-- fill in
        2018: None,   # <-- fill in
        2019: None,   # <-- fill in
        2020: None,   # <-- fill in
    }
    table_4_4_df = pd.DataFrame(list(table_4_4_dict.items()), columns=["Year", "Mean_VCI"])

# Sanity check: flag any missing/None values before merging
missing_vci = table_4_4_df[table_4_4_df["Mean_VCI"].isna()]
if len(missing_vci) > 0:
    print("\n*** WARNING: The following years are missing Mean_VCI values ***")
    print(missing_vci["Year"].tolist())
    print("Figure will be INCOMPLETE until these are filled in.\n")

# Merge — use LEFT join on loyo_df so missing years are visible as NaN, not silently dropped
merged = loyo_df.merge(table_4_4_df, left_on="Held_Out_Year", right_on="Year", how="left")

missing_after_merge = merged[merged["Mean_VCI"].isna()]
if len(missing_after_merge) > 0:
    print("*** WARNING: These LOYO years had NO match in Table 4.4 after merge ***")
    print(missing_after_merge["Held_Out_Year"].tolist())

print(f"\nMerged dataset shape: {merged.shape} (should be 20 rows if complete)")
print(merged[["Held_Out_Year", "Mean_Bias", "Mean_VCI"]])

drought_years = [2001, 2005, 2006]
merged["highlight"] = merged["Held_Out_Year"].isin(drought_years)

merged_csv = os.path.join(output_dir, "LOYO_merged_with_Table4_4.csv")
merged.to_csv(merged_csv, index=False)
print(f"\nSaved merged dataset: {merged_csv}")


# ======================================================================
# STEP 8: COMBINED FIGURE — Panel (a) R² per year + Panel (b) Bias vs Severity
# ======================================================================
print("\n" + "="*90)
print("STEP 8: Generating combined Figure 4.21 (R² + Bias-vs-Severity)")
print("="*90)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), facecolor="white")

# ---------------- Panel (a): R² per held-out year ----------------
ax = axes[0]
mean_r2 = loyo_df["R2"].mean()
colors_a = [LOYO_COLOR if r2 >= mean_r2 else "#D9A6AD" for r2 in loyo_df["R2"]]

bars = ax.bar(
    loyo_df["Held_Out_Year"].astype(str),
    loyo_df["R2"],
    color=colors_a,
    edgecolor="black",
    linewidth=0.6,
    alpha=0.9,
    width=0.65
)
ax.axhline(0, color="#999999", lw=1.0, zorder=0)
ax.axhline(mean_r2, color="#333333", lw=1.2, ls="--",
           label=f"Mean R² = {mean_r2:.3f} ± {loyo_df['R2'].std():.3f}")

ax.set_xlabel("Held-Out Year", fontweight="bold")
ax.set_ylabel("R² (Held-Out Year)", fontweight="bold")
ax.set_title("(a) LOYO R² per Held-Out Year", fontweight="bold", pad=10)
ax.tick_params(axis='x', rotation=45)
ax.legend(loc="lower right", frameon=False)

for bar, r2 in zip(bars, loyo_df["R2"]):
    va = "bottom" if r2 >= 0 else "top"
    offset = 0.02 if r2 >= 0 else -0.02
    ax.text(bar.get_x() + bar.get_width()/2, r2 + offset,
             f"{r2:.2f}", ha="center", va=va, fontsize=7.5)

# ---------------- Panel (b): Bias vs. VCI severity scatter ----------------
ax = axes[1]

plot_data = merged.dropna(subset=["Mean_VCI", "Mean_Bias"])
non_drought = plot_data[~plot_data["highlight"]]
drought = plot_data[plot_data["highlight"]]

ax.scatter(non_drought["Mean_VCI"], non_drought["Mean_Bias"],
           color="#A9A9A9", edgecolor="black", linewidth=0.5, s=70,
           label="Other years", zorder=2)
ax.scatter(drought["Mean_VCI"], drought["Mean_Bias"],
           color=LOYO_COLOR, edgecolor="black", linewidth=0.9, s=130,
           marker="D", label="Representative drought years", zorder=3)

for _, row in drought.iterrows():
    ax.annotate(str(int(row["Held_Out_Year"])),
                (row["Mean_VCI"], row["Mean_Bias"]),
                textcoords="offset points", xytext=(7, 7),
                fontsize=9.5, fontweight="bold")

# Trend line (only if enough valid points exist)
if len(plot_data) >= 3:
    from scipy.stats import linregress
    slope, intercept, r_value, p_value, std_err = linregress(
        plot_data["Mean_VCI"], plot_data["Mean_Bias"]
    )
    x_line = np.linspace(plot_data["Mean_VCI"].min(), plot_data["Mean_VCI"].max(), 100)
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, color="#333333", lw=1.3, ls="--", zorder=1,
            label=f"Trend (r = {r_value:.2f}, p = {p_value:.3f})")
    print(f"\nBias vs. Severity trend: slope={slope:.4f}, r={r_value:.3f}, p={p_value:.4f}")
else:
    print("\nNot enough complete data points to compute trend line — fill in Table 4.4 values.")

ax.axhline(0, color="#999999", lw=1.0, ls="-")
ax.set_xlabel("Mean VCI (Severity, Table 4.4)", fontweight="bold")
ax.set_ylabel("Mean Bias (Predicted − Observed VCI)", fontweight="bold")
ax.set_title("(b) Bias vs. Drought Severity", fontweight="bold", pad=10)
ax.legend(loc="upper right", frameon=False, fontsize=9)

plt.tight_layout()
combined_plot = os.path.join(output_dir, "Figure_4_21_LOYO_Combined_R2_BiasSeverity.png")
plt.savefig(combined_plot)
plt.close()
print(f"\nSaved combined figure: {combined_plot}")


# ======================================================================
# STEP 9: TABLE 4.12 — COMPACT MAIN-TEXT SUMMARY TABLE
# ======================================================================
print("\n" + "="*90)
print("STEP 9: Generating Table 4.12 (compact summary) and Appendix Table A3 (full)")
print("="*90)

# ---- Table 4.12: Mean ± SD summary (goes in main Results text) ----
table_4_12 = pd.DataFrame({
    "Metric": ["R²", "RMSE", "MAE", "Pearson r", "Spearman ρ", "Mean Bias"],
    "Mean": [
        loyo_df["R2"].mean(),
        loyo_df["RMSE"].mean(),
        loyo_df["MAE"].mean(),
        loyo_df["Pearson_r"].mean(),
        loyo_df["Spearman_rho"].mean(),
        loyo_df["Mean_Bias"].mean(),
    ],
    "SD": [
        loyo_df["R2"].std(),
        loyo_df["RMSE"].std(),
        loyo_df["MAE"].std(),
        loyo_df["Pearson_r"].std(),
        loyo_df["Spearman_rho"].std(),
        loyo_df["Mean_Bias"].std(),
    ],
})
table_4_12["Mean ± SD"] = table_4_12.apply(
    lambda r: f"{r['Mean']:.3f} ± {r['SD']:.3f}", axis=1
)
table_4_12_display = table_4_12[["Metric", "Mean ± SD"]]

table_4_12_csv = os.path.join(output_dir, "Table_4_12_LOYO_Summary.csv")
table_4_12_display.to_csv(table_4_12_csv, index=False)

print("\n--- TABLE 4.12: LOYO Cross-Validation Summary (paste into main text) ---")
print(table_4_12_display.to_string(index=False))
print(f"\nSaved: {table_4_12_csv}")

# ---- Also save as a clean image (optional, Q1-style table figure) ----
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.axis("off")
tbl = ax.table(
    cellText=table_4_12_display.values,
    colLabels=table_4_12_display.columns,
    cellLoc="center",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.6)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight="bold")
        cell.set_facecolor("#B23A48")
        cell.set_text_props(color="white")
    cell.set_edgecolor("#CCCCCC")
plt.tight_layout()
table_4_12_png = os.path.join(output_dir, "Table_4_12_LOYO_Summary.png")
plt.savefig(table_4_12_png, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved table image: {table_4_12_png}")


# ======================================================================
# APPENDIX TABLE A3 — FULL 20-ROW PER-YEAR RESULTS
# ======================================================================
appendix_table_a3 = loyo_df.copy()
appendix_table_a3 = appendix_table_a3.round({
    "R2": 3, "RMSE": 4, "MAE": 4,
    "Pearson_r": 3, "Spearman_rho": 3, "Mean_Bias": 4
})
appendix_table_a3 = appendix_table_a3.rename(columns={
    "Held_Out_Year": "Year",
    "N_Test_Samples": "N",
    "R2": "R²",
    "Pearson_r": "Pearson r",
    "Spearman_rho": "Spearman ρ",
    "Mean_Bias": "Bias"
})

appendix_csv = os.path.join(output_dir, "Appendix_Table_A3_LOYO_FullResults.csv")
appendix_table_a3.to_csv(appendix_csv, index=False)

print("\n--- APPENDIX TABLE A3: Full LOYO Results per Year (paste into Appendix) ---")
print(appendix_table_a3.to_string(index=False))
print(f"\nSaved: {appendix_csv}")

# ---- Also save Appendix Table A3 as a clean image ----
fig, ax = plt.subplots(figsize=(10, 7))
ax.axis("off")
tbl = ax.table(
    cellText=appendix_table_a3.values,
    colLabels=appendix_table_a3.columns,
    cellLoc="center",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.3)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight="bold", color="white")
        cell.set_facecolor("#1F6E8C")
    cell.set_edgecolor("#CCCCCC")
plt.tight_layout()
appendix_png = os.path.join(output_dir, "Appendix_Table_A3_LOYO_FullResults.png")
plt.savefig(appendix_png, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved appendix table image: {appendix_png}")

print("\n" + "="*90)
print("ALL DELIVERABLES GENERATED:")
print(" 1. Figure_4_21_LOYO_Combined_R2_BiasSeverity.png   <- the +1 figure")
print(" 2. Table_4_12_LOYO_Summary.csv / .png              <- the +1 main-text table")
print(" 3. Appendix_Table_A3_LOYO_FullResults.csv / .png   <- the +1 appendix table")
print(f"\nSaved Location: {output_dir}")
print("="*90)


STEP 7: Merging LOYO results with Table 4.4 VCI severity values
Loaded Table 4.4 from file: E:\2_Contribution\Final\Preprocess\3. Trend\Update result\Table_4_4_recomputed.csv

Merged dataset shape: (20, 15) (should be 20 rows if complete)
    Held_Out_Year  Mean_Bias  Mean_VCI
0            2001   0.230570     16.69
1            2002   0.108410     39.72
2            2003   0.071111     46.81
3            2004   0.007645     40.49
4            2005   0.113518     27.43
5            2006   0.136493     26.34
6            2007   0.077818     42.32
7            2008  -0.006298     38.54
8            2009  -0.075272     40.62
9            2010  -0.008515     47.96
10           2011  -0.122212     43.95
11           2012  -0.127585     65.80
12           2013  -0.131348     59.16
13           2014  -0.033032     51.55
14           2015  -0.038809     49.43
15           2016  -0.048718     52.11
16           2017   0.024027     52.21
17           2018  -0.130958     70.67
18           2019  

In [10]:
# Sensitivity check: does the relationship hold without 2001 anchoring the extreme end?
without_2001 = merged[merged["Held_Out_Year"] != 2001]
slope2, intercept2, r_value2, p_value2, std_err2 = linregress(
    without_2001["Mean_VCI"], without_2001["Mean_Bias"]
)
print(f"Without 2001: r = {r_value2:.3f}, p = {p_value2:.4f}")

Without 2001: r = -0.807, p = 0.0000
